# 29 Sensitivitas Jumlah Cluster k


In [1]:
from pathlib import Path
import os, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
ROOT_29 = Path(os.environ.get('SENSITIVITY29_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_29 / 'LIB').is_dir():
    raise FileNotFoundError('Folder LIB proyek belum tersedia: ' + str(ROOT_29))
print('Proyek:', ROOT_29)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Desain dan pemakaian ulang

| Konteks | Matriks tetap | Jumlah partisi |
|---|---|---|
| LMS full | 6.658 peserta, 13 fitur | 5 |
| LMS resampling | 30 indeks 80% yang sama, 5.326 peserta per replikasi | 150 |
| Pendikom train | 322 peserta, 18 fitur | 5 |
| Pendikom test | 139 peserta; assignment ke mode train, tanpa fitting | 0 |

Total 160 konteks-k. Jika versi Python dan pustaka ilmiah sama dengan tahap
25, 32 partisi/profil k=16 digunakan ulang setelah verifikasi isi; paling
banyak 128 fitting baru. Jika lingkungan berbeda, k=16 ikut dihitung ulang
dalam lingkungan sekarang agar perbandingan antar k konsisten (160 fitting).
Artefak lama tetap disimpan pada tahap asal. Tidak ada pengulangan praproses,
split, delapan initializer, classifier, atau sepuluh benchmark.

Rule utama tetap **mode-first/member-tie**, dengan magnitude dari bukti
anggota sesuai tahap 26. Fungsi diperluas hanya agar banyaknya baris cluster
mengikuti k. Profil unresolved tetap dibedakan dari skala Balanced 1/3.
Cluster berbeda boleh memetakan ke Base LS yang sama. Kegagalan fitting
dilaporkan terpisah; tidak disamarkan sebagai unresolved dan tidak diganti
dengan seed/metode lain.


In [2]:
KONFIGURASI_29 = {
    'k_values': [8, 12, 16, 20, 24],
    'resample_runs': 30,
    'reuse_k16_if_compatible': True,
    'save_member_comparisons': True,
    'force': False,
}


## Ukuran dan ketidakpastian

- Setiap k: cost, DBI/SI berbasis SMD, ukuran cluster, unresolved per dimensi
  dan peserta, jumlah/diversitas Base LS teramati, serta konflik mode-anggota.
- LMS: bandingkan setiap resample dengan model full **pada k yang sama**;
  laporkan ARI/NMI/AMI, agreement empat arah dan Base LS. Ringkas 30 resample
  sebagai mean, SD, rentang, dan selisih berpasangan terhadap k=16. Resample
  bertumpang tindih, sehingga tidak diperlakukan sebagai cohort independen.
- Pendikom: profil test berasal dari mode/rule yang dibentuk pada train.
  Accuracy, balanced accuracy, dan macro-F1 per dimensi dan rata-rata empat
  dimensi; confusion matrices memasukkan unresolved sebagai abstention/error.
- Gunakan kembali 10.000 bobot bootstrap peserta tahap 27. Peserta, empat
  dimensi, serta semua k tetap berpasangan. CI individual 95% deskriptif;
  empat kontras utama mean BA (k8/12/20/24 minus k16) juga memiliki interval
  percentile Bonferroni keluarga empat. Tidak ada pemilihan k dari test.

CI bersyarat pada split dan rule tetap, bukan bukti cohort eksternal baru.
Agreement lintas-k terhadap k=16 merupakan sensitivitas, bukan akurasi ILS.
Identitas sampel SI tetap per konteks untuk seluruh k. Hasil waktu cached
tidak diperlakukan sebagai waktu fitting baru yang sebanding.

Cache memeriksa isi masukan, kode, konfigurasi, versi, dan hasil per partisi,
profil, stabilitas, serta evaluasi. Jika Colab terputus, jalankan notebook
yang sama kembali. Perubahan tampilan output notebook tidak membatalkan cache.
`force=True` menghitung ulang artefak lokal tahap 29; pemakaian ulang sumber
k16 tetap mengikuti opsi terpisah. Untuk menghitung ulang seluruh k, gunakan
`reuse_k16_if_compatible=False` dan `force=True`.


In [3]:
HELPER26_SOURCE = '"""Fixed-rule semantic evaluation from the immutable partition bank of stage 25."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport io\nimport itertools\nimport json\nimport os\nimport platform\nimport re\nimport sys\nimport uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE = \'26_Pengaruh_Initializer_Profil_Semantik\'\nUPSTREAM = \'25_Perbandingan_Initializer_Terintegrasi\'\nUP = \'OUTPUT/TAMBAHAN/\' + UPSTREAM\nUP_CONTROL = \'OUTPUT/TAMBAHAN/KONTROL/\' + UPSTREAM\nMETHODS = [\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\nSTOCHASTIC = {\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\nORDER = [\'CARE-KM\',\'Cao\',\'Bai\',\'BK-Modes\',\'K-Modes++\',\'Random\',\'Huang\',\'HR-2023\']\nKEYS = [\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\nDIMS = [\'Active/Reflective\',\'Sensing/Intuitive\',\'Visual/Verbal\',\'Sequential/Global\']\nPOLES = [(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\nABBR = {\'Active\':\'Act.\',\'Reflective\':\'Ref.\',\'Sensing\':\'Sens.\',\'Intuitive\':\'Int.\',\n        \'Visual\':\'Vis.\',\'Verbal\':\'Ver.\',\'Sequential\':\'Seq.\',\'Global\':\'Gl.\'}\nLS_LOOKUP = {combo:\'LS%02d\' % (i+1) for i,combo in enumerate(itertools.product(*POLES))}\nMAIN = [\'content_visit\',\'content_stay\',\'selfass_visit_proxy\',\'selfass_stay_proxy\',\'exercise_visit_proxy\',\n        \'exercise_stay_proxy\',\'forum_visit\',\'forum_stay\',\'forum_post\',\'quiz_stay_results_proxy\',\n        \'navigation_overview_visit\',\'navigation_overview_stay\',\'return_to_overview_rate\']\nPEND = [\'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\'exercise_visit\',\'exercise_stay\',\n        \'ques_detail\',\'ques_overview\',\'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\'ques_interpret\',\n        \'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\'navigation_overview_visit\',\'navigation_overview_stay\']\nFULL = \'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv\'\nRULE_FILES = {\'LMS\':\'OUTPUT/EKSPERIMEN/SEMANTIK/11_Interpretasi_Semantik_FSLSM/08_semantic_evidence_matrix.csv\',\n              \'PENDIKOM\':\'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_semantic_rules.csv\'}\nDEFAULT_CONFIG = {\'bootstrap_repetitions\':10000,\'bootstrap_seed\':20261014,\n                  \'save_member_comparisons\':True,\'force\':False}\nMETRIC_NAMES = [\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(value):\n    return sha(json.dumps(clean(value),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode(\'utf-8\'))\n\n\ndef clean(value):\n    if isinstance(value,dict): return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple,np.ndarray)): return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)): return int(value)\n    if isinstance(value,(float,np.floating)): return float(value) if np.isfinite(value) else None\n    if isinstance(value,np.bool_): return bool(value)\n    return value\n\n\ndef atomic(path, data):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp = path.with_name(path.name+\'.\'+uuid.uuid4().hex[:8]+\'.tmp\')\n    temp.write_bytes(data); temp.replace(path)\n\n\ndef save_json(path, value):\n    atomic(path,json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\'))\n\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False,lineterminator=\'\\n\',float_format=\'%.17g\').encode(\'utf-8\')\n    atomic(path,gzip.compress(data,mtime=0) if str(path).endswith(\'.gz\') else data)\n\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        return obj if isinstance(obj,dict) else {}\n    except (OSError,ValueError,UnicodeError): return {}\n\n\nclass Files:\n    def __init__(self, root):\n        self.root = Path(root).resolve(); self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root): raise ValueError(\'Lokasi di luar proyek: \'+relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative); before = path.stat()\n        stamp = (before.st_size,before.st_mtime_ns,before.st_ctime_ns)\n        if relative in self.cache and self.cache[relative][0] == stamp: return self.cache[relative][1]\n        data = path.read_bytes(); after = path.stat()\n        if stamp != (after.st_size,after.st_mtime_ns,after.st_ctime_ns):\n            raise RuntimeError(\'Berkas berubah ketika dibaca: \'+relative)\n        self.cache[relative] = (stamp,data); return data\n\n    def hash(self, relative): return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith(\'.gz\'): data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data),dtype=str if strings else None,\n                           keep_default_na=not strings,float_precision=\'round_trip\')\n\n\ndef runtime():\n    packages = {}\n    for name in [\'numpy\',\'pandas\',\'scipy\',\'sklearn\',\'matplotlib\']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod,\'__version__\',None) or importlib.metadata.version(\'scikit-learn\' if name==\'sklearn\' else name)\n    return {\'python\':platform.python_version(),\'packages\':packages}\n\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\').decode(\'utf-8-sig\')); code=[]\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']!=\'code\': continue\n        source=\'\'.join(cell[\'source\'])\n        if tag in cell.get(\'metadata\',{}).get(\'tags\',[]):\n            tree=ast.parse(source)\n            if (len(tree.body)!=1 or not isinstance(tree.body[0],ast.Assign)\n                or len(tree.body[0].targets)!=1 or not isinstance(tree.body[0].targets[0],ast.Name)\n                or tree.body[0].targets[0].id!=variable or not isinstance(ast.literal_eval(tree.body[0].value),dict)):\n                raise RuntimeError(\'Sel konfigurasi hanya boleh berisi literal \'+variable)\n        else: code.append(source)\n    return digest(code)\n\n\ndef expected_rules(dataset):\n    rows=[]\n    def add(features,dim,low,high):\n        for f in features.split(): rows.append((f,dim,low,high))\n    if dataset==\'LMS\':\n        add(\'content_visit content_stay selfass_stay_proxy forum_visit quiz_stay_results_proxy\',DIMS[0],*POLES[0])\n        add(\'selfass_visit_proxy exercise_visit_proxy exercise_stay_proxy forum_post\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay\',DIMS[1],*POLES[1])\n        add(\'selfass_visit_proxy selfass_stay_proxy exercise_visit_proxy quiz_stay_results_proxy\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit forum_visit forum_stay forum_post\',DIMS[2],*POLES[2])\n        add(\'navigation_overview_visit navigation_overview_stay return_to_overview_rate\',DIMS[3],*POLES[3])\n    else:\n        add(\'selfass_visit selfass_twice_wrong exercise_visit exercise_stay\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay quiz_stay_results\',DIMS[0],*POLES[0])\n        add(\'selfass_visit exercise_visit ques_detail ques_facts quiz_revisions quiz_stay_results\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit content_stay ques_concepts ques_develop\',DIMS[1],*POLES[1])\n        add(\'ques_graphics\',DIMS[2],*POLES[2][::-1]); add(\'content_visit ques_text\',DIMS[2],*POLES[2])\n        add(\'ques_detail\',DIMS[3],*POLES[3][::-1])\n        add(\'ques_overview ques_interpret ques_develop navigation_overview_visit navigation_overview_stay\',DIMS[3],*POLES[3])\n    return sorted(rows)\n\n\ndef check_config(configuration):\n    if set(configuration)-set(DEFAULT_CONFIG): raise ValueError(\'Konfigurasi tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration)\n    for key in [\'bootstrap_repetitions\',\'bootstrap_seed\']:\n        if isinstance(cfg[key],bool) or not isinstance(cfg[key],int) or cfg[key]<0: raise ValueError(key+\' tidak valid.\')\n    if cfg[\'bootstrap_repetitions\']<100: raise ValueError(\'Gunakan sedikitnya 100 bootstrap untuk diagnostik; default publikasi 10000.\')\n    for key in [\'save_member_comparisons\',\'force\']:\n        if not isinstance(cfg[key],bool): raise ValueError(key+\' harus True/False.\')\n    return cfg\n\n\ndef input_setup(files):\n    required = [UP+\'/PROTOKOL/02_konfigurasi.json\',UP_CONTROL+\'/status_eksekusi.json\',\n                FULL,*RULE_FILES.values(),\'LIB/pipeline_data.py\',\'LIB/pendikom_ils_labels.csv\',\n                \'OUTPUT/SPLIT/LMS/train_categorizer_state.json\',\n                \'OUTPUT/SPLIT/LMS/70_30_main_lms_train_only_thresholds.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\',\n                \'PYTHON/\'+UPSTREAM+\'.ipynb\',\'PYTHON/\'+STAGE+\'.ipynb\']\n    for ds in [\'LMS\',\'PENDIKOM\']:\n        required += [f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n    required += [f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv\' for r in range(30)]\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing: raise FileNotFoundError(\'Masukan belum tersedia. Selesaikan tahap 25 dan pertahankan hasil 01–24:\\n\'+\'\\n\'.join(missing[:20]))\n    status=json.loads(files.read(UP_CONTROL+\'/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'full_protocol\') is not True or status.get(\'n_valid\')!=3272:\n        raise RuntimeError(\'Tahap 25 belum selesai dengan protokol lengkap dan seluruh run valid.\')\n    protocol=json.loads(files.read(UP+\'/PROTOKOL/02_konfigurasi.json\'))\n    if protocol[\'constants\'] != {\'FROZEN_CARE_CONFIG\':{\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0},\'K\':16,\'MAX_ITER\':100,\'SI_SAMPLE_SIZE\':2000}:\n        raise RuntimeError(\'Konfigurasi tahap 25 berbeda dari protokol yang ditetapkan.\')\n    source_code=protocol[\'code\']\n    for relative,checksum in source_code[\'project_libraries\'].items():\n        if files.hash(relative)!=checksum: raise RuntimeError(\'Pustaka sumber tahap 25 berubah: \'+relative)\n        required.append(relative)\n    if notebook_hash(files,UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\') != source_code[\'notebook_code\']:\n        raise RuntimeError(\'Kode notebook 25 berubah setelah eksperimen. Gunakan notebook sumber yang sesuai.\')\n    literals={}\n    nb=json.loads(files.read(\'PYTHON/\'+UPSTREAM+\'.ipynb\'))\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']==\'code\':\n            for node in ast.parse(\'\'.join(cell[\'source\'])).body:\n                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):\n                    literals[node.targets[0].id]=node.value.value\n    for name,var in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if sha(literals[var].encode(\'utf-8\'))!=source_code[\'embedded_sources\'][name]: raise RuntimeError(\'Sumber tertanam tahap 25 tidak sesuai.\')\n    feature_state={}\n    for node in ast.parse(files.read(\'LIB/pipeline_data.py\').decode(\'utf-8-sig\')).body:\n        if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and node.targets[0].id in [\'MAIN_FEATURES\',\'PENDIKOM_FEATURES\']:\n            feature_state[node.targets[0].id]=ast.literal_eval(node.value)\n    if feature_state[\'MAIN_FEATURES\']!=MAIN or [x for x in feature_state[\'PENDIKOM_FEATURES\'] if x!=\'navigation_skip\']!=PEND:\n        raise RuntimeError(\'Urutan fitur tidak sesuai protokol.\')\n    audit=files.frame(\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',True)\n    if audit.loc[audit.used.str.lower().eq(\'true\'),\'feature\'].tolist()!=PEND: raise RuntimeError(\'Mask fitur train Pendikom berubah.\')\n    rules={}\n    for ds,path in RULE_FILES.items():\n        frame=files.frame(path,True)\n        cols=[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']\n        if sorted(map(tuple,frame[cols].to_numpy())) != expected_rules(ds): raise RuntimeError(\'Evidence rules berbeda dari aturan utama: \'+ds)\n        if \'moderate_supports\' in frame and not frame.moderate_supports.eq(\'Neutral\').all(): raise RuntimeError(\'Evidence moderate harus netral.\')\n        rules[ds]=frame[cols]\n    watched={p:files.hash(p) for p in set(required) if not p.endswith(\'.ipynb\') and not p.endswith(\'status_eksekusi.json\')}\n    return protocol,rules,watched\n\n\ndef contexts(files):\n    def make(scope,ds,frame,features,idcol,source,rep=None,idx=None,test=None):\n        if frame[idcol].duplicated().any(): raise RuntimeError(\'Identitas peserta tidak unik: \'+scope)\n        X=frame[features].to_numpy(dtype=object)\n        if not np.isin(X,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori di luar 1/2/3: \'+scope)\n        indices=np.arange(len(X),dtype=np.int64) if idx is None else idx\n        ids=frame[idcol].tolist();testids=test[idcol].tolist() if test is not None else []\n        if len(set(testids))!=len(testids) or set(ids)&set(testids): raise RuntimeError(\'Train/test tumpang tindih atau ID test berulang.\')\n        Xt=test[features].to_numpy(dtype=object) if test is not None else None\n        matrix=digest({\'features\':features,\'rows\':X.tolist(),\'ids\':ids,\'indices\':indices.tolist(),\n                       \'test_rows\':Xt.tolist() if Xt is not None else None,\'test_ids\':testids,\'classes\':None})\n        return {\'scope\':scope,\'dataset\':ds,\'X\':X,\'ids\':ids,\'indices\':indices,\'test_X\':Xt,\'test_ids\':testids,\n                \'features\':features,\'sources\':source,\'rep\':rep,\'matrix_sha256\':matrix,\'test_frame\':test}\n    frame=files.frame(FULL,True)\n    if frame.shape!=(6658,14) or frame.columns.tolist()!=[\'user\']+MAIN: raise RuntimeError(\'Matriks full LMS berbeda.\')\n    yield make(\'LMS_FULL\',\'LMS\',frame,MAIN,\'user\',[FULL])\n    for rep in range(30):\n        path=f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv\'\n        idx=files.frame(path)[\'source_row_index\'].to_numpy(np.int64)\n        if len(idx)!=5326 or len(np.unique(idx))!=5326 or np.any(idx<0) or np.any(idx>=6658): raise RuntimeError(\'Indeks resample tidak valid.\')\n        yield make(\'LMS_RESAMPLING\',\'LMS\',frame.iloc[idx],MAIN,\'user\',[FULL,path],rep,idx)\n    for ds,ntr,nte,features,idcol in [(\'LMS\',4660,1998,MAIN,\'user\'),(\'PENDIKOM\',322,139,PEND,\'userid\')]:\n        paths=[f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n        tr,te=[files.frame(p,True) for p in paths]\n        if len(tr)!=ntr or len(te)!=nte: raise RuntimeError(\'Ukuran train/test berbeda: \'+ds)\n        if \'split\' in tr and (not tr.split.str.lower().eq(\'train\').all() or not te.split.str.lower().eq(\'test\').all()): raise RuntimeError(\'Kolom split tidak sesuai.\')\n        yield make(ds+\'_TRAIN\',ds,tr,features,idcol,paths,test=te)\n\n\ndef job_identity(context,method,run):\n    return {\'scope\':context[\'scope\'],\'dataset\':context[\'dataset\'],\'rep\':context[\'rep\'],\'run\':run,\'method\':method}\n\n\ndef upstream_job(files,protocol,context,method,run):\n    identity=job_identity(context,method,run);key=digest(identity)[:24]\n    relative=UP_CONTROL+\'/JOBS/\'+key+\'.json\'\n    if not files.path(relative).is_file(): raise FileNotFoundError(\'Partisi tahap 25 belum tersedia: \'+str(identity))\n    record=json.loads(files.read(relative));checksum=record.pop(\'record_sha256\',\'\')\n    if digest(record)!=checksum or record.get(\'status\')!=\'selesai\' or record.get(\'identity\')!=identity:\n        raise RuntimeError(\'Catatan partisi tahap 25 tidak valid: \'+relative)\n    p=record[\'parameters\'];row=record[\'row\']\n    if record[\'signature\']!=digest({\'parameters\':p,\'code\':protocol[\'code\'],\'runtime\':protocol[\'runtime\']}): raise RuntimeError(\'Signature tahap 25 tidak sesuai protokol.\')\n    expected_seed=(6161600+context[\'rep\']) if context[\'rep\'] is not None else 4161600+run\n    if (p[\'matrix_sha256\']!=context[\'matrix_sha256\'] or p[\'k\']!=16 or p[\'seed\']!=expected_seed\n        or p[\'max_iter\']!=100 or p[\'care_config\']!=protocol[\'constants\'][\'FROZEN_CARE_CONFIG\']\n        or not p[\'needs_partition\'] or row.get(\'valid\') is not True or not row.get(\'partition_available\')):\n        raise RuntimeError(\'Isi matriks/parameter/partisi tahap 25 tidak sesuai: \'+str(identity))\n    watched={relative:files.hash(relative)}\n    for path,expected in record[\'outputs\'].items():\n        if not path.startswith(UP+\'/BANK/\'): raise RuntimeError(\'Lokasi bank tahap 25 tidak sesuai.\')\n        if files.hash(path)!=expected: raise RuntimeError(\'Artefak tahap 25 berubah. Pulihkan/jalankan ulang 25: \'+path)\n        watched[path]=expected\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1: raise RuntimeError(\'Artefak wajib tidak unik: \'+filename)\n        return found[0]\n    modes=files.frame(find(\'final_modes.csv\'),True)\n    if modes.columns.tolist()!=[\'cluster_id\']+context[\'features\'] or modes.cluster_id.astype(int).tolist()!=list(range(16)):\n        raise RuntimeError(\'Skema/urutan mode akhir tidak sesuai.\')\n    modes=modes[context[\'features\']].to_numpy(dtype=object)\n    if not np.isin(modes,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori mode tidak valid.\')\n    assignment=files.frame(find(\'train_assignments.csv\'),True)\n    if assignment.identity.tolist()!=context[\'ids\'] or not np.array_equal(assignment.source_row_index.astype(np.int64),context[\'indices\']):\n        raise RuntimeError(\'Identitas/urutan anggota tidak sesuai matriks.\')\n    labels=assignment.cluster_id.to_numpy(int)\n    if not np.isin(labels,np.arange(16)).all() or (np.bincount(labels,minlength=16)==0).any(): raise RuntimeError(\'Keanggotaan cluster tidak valid.\')\n    cost=int(np.sum(context[\'X\']!=modes[labels]))\n    if cost!=int(row[\'final_cost\']): raise RuntimeError(\'Cost anggota/mode berbeda dari hasil tahap 25.\')\n    testlabels=None\n    if context[\'test_X\'] is not None:\n        test=files.frame(find(\'test_assignments.csv\'),True)\n        if test.identity.tolist()!=context[\'test_ids\']: raise RuntimeError(\'Urutan identitas test berbeda.\')\n        testlabels=test.cluster_id.to_numpy(int)\n        distance=np.zeros((len(testlabels),16),dtype=np.int64)\n        for j in range(len(context[\'features\'])): distance += context[\'test_X\'][:,j,None]!=modes[None,:,j]\n        if not np.array_equal(testlabels,np.argmin(distance,axis=1)) or not np.array_equal(test.distance.astype(int),distance.min(axis=1)):\n            raise RuntimeError(\'Assignment test berbeda dari mode train yang dibekukan.\')\n    return {\'record\':record,\'checksum\':checksum,\'watched\':watched,\'modes\':modes,\'labels\':labels,\'test_labels\':testlabels,\'identity\':identity}\n\n\ndef interpret(X,modes,labels,features,rules):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    sizes=np.bincount(labels,minlength=16);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(16):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n\n\ndef profiles(mapping,labels,ids,indices=None):\n    cols=KEYS+[k+\'_score\' for k in KEYS]+[\'ls_class\',\'mapping_status\']\n    frame=mapping.set_index(\'cluster_id\').loc[labels,cols].reset_index()\n    frame.insert(0,\'identity\',ids)\n    if indices is not None: frame.insert(1,\'source_row_index\',indices)\n    return frame\n\n\ndef numeric_profiles(frame):\n    return np.column_stack([np.where(frame[k].eq(ABBR[p[0]]),1,np.where(frame[k].eq(ABBR[p[1]]),-1,0)) for k,p in zip(KEYS,POLES)])\n\n\ndef cache_read(files,path,signature,force=False):\n    record=read_checkpoint(path)\n    checksum=record.pop(\'record_sha256\',\'\')\n    try: valid=digest(record)==checksum\n    except (TypeError,ValueError): valid=False\n    if force or not valid or record.get(\'signature\')!=signature or record.get(\'status\')!=\'selesai\': return None\n    if not all(files.path(p).is_file() and files.hash(p)==h for p,h in record.get(\'outputs\',{}).items()): return None\n    return record\n\n\ndef cache_write(files,path,signature,outputs,extra):\n    record={\'status\':\'selesai\',\'signature\':signature,\'outputs\':{str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in outputs},**extra}\n    record[\'record_sha256\']=digest(record);save_json(path,record);return record\n\n\ndef semantic_job(files,output,control,context,upstream,rules,code_state,cfg):\n    identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream_record_sha256\':upstream[\'checksum\'],\n                      \'matrix_sha256\':context[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    checkpoint=control/\'PROFIL\'/f\'{key}.json\'\n    record=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    folder=output/\'BANK\'/identity[\'scope\']/identity[\'dataset\']/(\'r%02d\' % (identity[\'rep\'] if identity[\'rep\'] is not None else identity[\'run\']))/re.sub(r\'[^A-Za-z0-9]+\',\'_\',identity[\'method\'].replace(\'++\',\'_PP\')).strip(\'_\')\n    mapping_path=folder/\'01_cluster_profiles.csv\';evidence_path=folder/\'02_cluster_dimension_evidence.csv\'\n    if record:\n        mapping=files.frame(str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'));evidence=files.frame(str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\'))\n        disposition=\'CACHE\'\n    else:\n        mapping,evidence=interpret(context[\'X\'],upstream[\'modes\'],upstream[\'labels\'],context[\'features\'],rules)\n        save_csv(mapping_path,mapping);save_csv(evidence_path,evidence);paths=[mapping_path,evidence_path]\n        if context[\'test_X\'] is not None:\n            for filename,frame in [(\'03_train_targets.csv\',profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])),\n                                   (\'04_test_targets.csv\',profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\']))]:\n                path=folder/filename;save_csv(path,frame);paths.append(path)\n        record=cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'upstream_artifacts\':upstream[\'watched\']})\n        disposition=\'MAP\'\n    weighted=sum(mapping.cluster_size*mapping.mapping_status.ne(\'RESOLVED\'))/len(context[\'X\'])\n    summary={**identity,\'n_learners\':len(context[\'X\']),\'n_clusters\':16,\'n_resolved_profiles\':mapping.loc[mapping.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique(),\n             \'unresolved_learner_rate\':weighted,\'mode_member_conflicts\':int((~evidence.mode_member_consistency.astype(bool)).sum()),\n             \'strong_conflicts\':int(evidence.strong_mode_member_conflict.astype(bool).sum()),\'mapping_file\':str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n             \'evidence_file\':str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\')}\n    return mapping,evidence,record,summary,disposition\n\n\ndef count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a,b,c,d,e,f=np.moveaxis(np.asarray(counts,float),-1,0)\n    pos=a+b+c;neg=d+e+f;total=pos+neg\n    with np.errstate(divide=\'ignore\',invalid=\'ignore\'):\n        accuracy=(a+e)/total;ba=.5*(a/pos+e/neg)\n        denp=2*a+d+b+c;denn=2*e+b+d+f\n        fp=np.divide(2*a,denp,out=np.zeros_like(a),where=denp>0)\n        fn=np.divide(2*e,denn,out=np.zeros_like(e),where=denn>0)\n    return np.stack([accuracy,ba,.5*(fp+fn)],axis=-1)\n\n\ndef confusion_masks(truth,prediction):\n    # prediction: run x learner x dimension; output learner x run x dimension x six cells.\n    t=truth[None,:,:];p=prediction\n    conditions=[(t==1)&(p==1),(t==1)&(p==-1),(t==1)&(p==0),\n                (t==-1)&(p==1),(t==-1)&(p==-1),(t==-1)&(p==0)]\n    return np.stack(conditions,axis=-1).transpose(1,0,2,3).astype(float)\n\n\ndef ils_reference(files,context):\n    path=\'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\';ref=files.frame(path,True)\n    test=context[\'test_frame\']\n    if ref.userid.tolist()!=context[\'test_ids\'] or ref.courseid.tolist()!=test.courseid.tolist(): raise RuntimeError(\'Reference ILS tidak sejajar dengan peserta test.\')\n    original=files.frame(\'LIB/pendikom_ils_labels.csv\',True).set_index([\'userid\',\'courseid\'])\n    original=original.loc[list(zip(ref.userid,ref.courseid))].reset_index()\n    cols=[\'ils_\'+k+\'_code\' for k in KEYS]+[\'ils_ls_class\']\n    if not original[cols].reset_index(drop=True).equals(ref[cols].reset_index(drop=True)) or not original.split.str.lower().eq(\'test\').all():\n        raise RuntimeError(\'Reference ILS test berbeda dari label independen yang dibekukan.\')\n    truth=[]\n    for key,poles in zip(KEYS,POLES):\n        column=ref[\'ils_\'+key+\'_code\'];first=column.str.startswith(ABBR[poles[0]]);second=column.str.startswith(ABBR[poles[1]])\n        if not (first^second).all(): raise RuntimeError(\'Label arah ILS tidak dikenal.\')\n        values=np.where(first,1,-1)\n        if len(np.unique(values))!=2: raise RuntimeError(\'Dua kutub reference harus terwakili untuk metrik BA.\')\n        truth.append(values)\n    return ref,np.column_stack(truth)\n\n\ndef evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg):\n    folder=output/\'ILS\';checkpoint=control/\'evaluasi_ils.json\'\n    signature=digest({\'code\':code_state,\'truth\':truth.tolist(),\'ids\':reference.userid.tolist(),\n        \'predictions\':{m:sha(np.asarray(predictions[m],dtype=np.int8).tobytes()) for m in METHODS},\n        \'bootstrap_repetitions\':cfg[\'bootstrap_repetitions\'],\'bootstrap_seed\':cfg[\'bootstrap_seed\'],\n        \'unit\':\'learner; all four dimensions and all runs/methods remain paired\'})\n    saved=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if saved:\n        print(\'Evaluasi ILS/CI: gunakan hasil yang isinya terverifikasi sama.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    n=len(truth);nr=cfg[\'bootstrap_repetitions\'];rng=np.random.default_rng(cfg[\'bootstrap_seed\'])\n    masks={m:confusion_masks(truth,np.asarray(predictions[m],dtype=np.int8)) for m in METHODS}\n    counts={m:masks[m].sum(axis=0) for m in METHODS}\n    points={m:count_metrics(counts[m]) for m in METHODS}\n    boot=np.empty((nr,len(METHODS),5,3),dtype=np.float64)\n    valid=np.zeros(nr,dtype=bool)\n    print(\'CI ILS:\',nr,\'bootstrap berpasangan pada\',n,\'peserta.\',flush=True)\n    with threadpool_limits(limits=1):\n        for start in range(0,nr,250):\n            stop=min(start+250,nr);weights=rng.multinomial(n,np.full(n,1/n),size=stop-start).astype(float)\n            positive=weights@(truth==1);negative=n-positive\n            valid[start:stop]=((positive>0)&(negative>0)).all(axis=1)\n            for mi,method in enumerate(METHODS):\n                source=masks[method];shape=source.shape\n                values=(weights@source.reshape(n,-1)).reshape(stop-start,shape[1],4,6)\n                metrics=count_metrics(values).mean(axis=1)\n                boot[start:stop,mi,:4,:]=metrics;boot[start:stop,mi,4,:]=metrics.mean(axis=1)\n    if valid.sum()<100: raise RuntimeError(\'Bootstrap valid kurang dari 100; interval belum dapat dihitung.\')\n    boot=boot[valid];paths=[];raw_rows=[];cm_rows=[];ref_rows=[];summary=[];overall=[];pred_rows=[]\n    actual_counts=np.stack([(truth==1).sum(axis=0),(truth==-1).sum(axis=0)],axis=-1)\n    for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n        for ci,pole in enumerate(poles): ref_rows.append({\'dimension\':key,\'reference_pole\':pole,\'n\':int(actual_counts[di,ci]),\'n_test\':n})\n    for mi,method in enumerate(METHODS):\n        arr=np.asarray(predictions[method],dtype=np.int8);point=points[method];nrun=len(arr)\n        for run in range(nrun):\n            for li,identity in enumerate(reference.userid):\n                row={\'method\':method,\'run\':run,\'seed\':4161600+run,\'identity\':identity,\'courseid\':reference.courseid.iloc[li],\n                     \'reference_ls_class\':reference.ils_ls_class.iloc[li]}\n                full=[]\n                for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1]\n                    value=arr[run,li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                    row[\'predicted_\'+key]=pole or \'UNRESOLVED\';full.append(pole)\n                row[\'predicted_ls_class\']=LS_LOOKUP[tuple(full)] if all(full) else \'AMBIGUOUS\';pred_rows.append(row)\n            for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                cc=counts[method][run,di];a,b,c,d,e,f=cc;resolved=n-c-f\n                raw_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'n_test\':n,\'reference_first\':int(actual_counts[di,0]),\n                                 \'reference_second\':int(actual_counts[di,1]),\'n_resolved\':int(resolved),\'n_unresolved\':int(c+f),\n                                 \'coverage\':resolved/n,\'resolved_only_accuracy\':(a+e)/resolved if resolved else np.nan,\n                                 **dict(zip(METRIC_NAMES,point[run,di]))})\n                for ci,num in enumerate(cc):\n                    cm_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            values=point[:,di,:] if di<4 else point.mean(axis=1)\n            row={\'method\':method,\'dimension\':key,\'n_runs\':nrun,\'n_test\':n,\n                 \'bootstrap_valid\':int(valid.sum()),\'bootstrap_requested\':nr,\n                 \'coverage_mean\':float((arr[:,:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                 \'complete_profile_coverage\':float((arr!=0).all(axis=2).mean())}\n            for ji,metric in enumerate(METRIC_NAMES):\n                low,high=np.quantile(boot[:,mi,di,ji],[.025,.975])\n                row[metric+\'_mean\']=float(values[:,ji].mean());row[metric+\'_sd_runs\']=float(values[:,ji].std(ddof=1)) if nrun>1 else np.nan\n                row[metric+\'_ci95_low\']=low;row[metric+\'_ci95_high\']=high\n            (summary if di<4 else overall).append(row)\n    differences=[];care=METHODS.index(\'CARE-KM\')\n    for mi,method in enumerate(METHODS):\n        if method==\'CARE-KM\': continue\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            for ji,metric in enumerate(METRIC_NAMES):\n                delta=boot[:,care,di,ji]-boot[:,mi,di,ji];lo,hi=np.quantile(delta,[.025,.975])\n                carepoint=points[\'CARE-KM\'][:,di,ji].mean() if di<4 else points[\'CARE-KM\'][:,:,ji].mean()\n                basepoint=points[method][:,di,ji].mean() if di<4 else points[method][:,:,ji].mean()\n                row={\'baseline\':method,\'dimension\':key,\'metric\':metric,\'care_minus_baseline\':carepoint-basepoint,\n                     \'paired_ci95_low\':lo,\'paired_ci95_high\':hi,\'positive_favors\':\'CARE-KM\',\'n_learners\':n,\n                     \'interval\':\'paired learner percentile bootstrap; individual 95% intervals\'}\n                if di==4 and metric==\'balanced_accuracy\':\n                    low,high=np.quantile(delta,[.05/(2*7),1-.05/(2*7)])\n                    row.update(primary_family=\'seven overall mean-BA contrasts\',bonferroni_family95_low=low,bonferroni_family95_high=high)\n                differences.append(row)\n    outputs=[(\'01_prediksi_per_peserta.csv\',pd.DataFrame(pred_rows)),(\'02_metrik_per_dimensi_per_run.csv\',pd.DataFrame(raw_rows)),\n             (\'03_ringkasan_per_dimensi.csv\',pd.DataFrame(summary)),(\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),\n             (\'05_confusion_matrices.csv\',pd.DataFrame(cm_rows)),(\'07_perbedaan_berpasangan_vs_care.csv\',pd.DataFrame(differences)),\n             (\'08_reference_counts.csv\',pd.DataFrame(ref_rows))]\n    for filename,frame in outputs:\n        path=folder/filename;save_csv(path,frame);paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,metrics=boot,methods=np.asarray(METHODS),dimensions=np.asarray(KEYS+[\'mean_four_dimensions\']),\n                                          metric_names=np.asarray(METRIC_NAMES),valid_draws=valid,seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'06_bootstrap_paired_metrics.npz\';atomic(path,buffer.getvalue());paths.append(path)\n    path=folder/\'09_protokol_evaluasi.json\'\n    save_json(path,{\'primary_metric\':\'mean balanced accuracy across four dimensions\',\'n_learners\':n,\'n_bootstrap_valid\':int(valid.sum()),\n        \'seed\':cfg[\'bootstrap_seed\'],\'unit\':\'learner; all dimensions, runs, and methods paired\',\n        \'stochastic_summary\':\'metric per run, then arithmetic mean; no best-run selection\',\n        \'interval_scope\':\'conditional on frozen training split, modes, mapping, and saved initialization runs\',\n        \'unresolved\':\'included as incorrect/abstention; two reference poles used for macro-F1; coverage separately reported\',\n        \'multiplicity\':\'individual paired CIs descriptive; primary seven overall mean-BA contrasts also have Bonferroni percentile family intervals\',\n        \'semantic_scope\':\'direction correspondence; behavioral 11-unit magnitudes are not questionnaire scores\'})\n    paths.append(path)\n    cache_write(files,checkpoint,signature,paths,{\'n_test\':n,\'n_methods\':8,\'bootstrap_valid\':int(valid.sum())})\n    return pd.DataFrame(overall),False\n\n\ndef stability_job(files,output,control,context,upstream,mapping,reference,code_state,cfg):\n    from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score,adjusted_mutual_info_score\n    method=upstream[\'identity\'][\'method\'];rep=context[\'rep\'];identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    path=control/\'STABILITAS\'/f\'{key}.json\';cached=cache_read(files,path,signature,cfg[\'force\'])\n    if cached: return cached[\'summary\'],\'CACHE\'\n    idx=context[\'indices\'];current=profiles(mapping,upstream[\'labels\'],context[\'ids\'],idx)\n    ref=reference[\'profiles\'].iloc[idx].reset_index(drop=True)\n    if ref.identity.tolist()!=context[\'ids\']: raise RuntimeError(\'Peserta resample tidak sejajar dengan full reference initializer.\')\n    truth=numeric_profiles(ref);pred=numeric_profiles(current);paired=(truth!=0)&(pred!=0);agreement=paired&(truth==pred)\n    complete=agreement.all(axis=1)\n    row={**identity,\'n_learners\':len(idx),\'reference_full_run\':0,\'reference_seed\':4161600,\n        \'mean_direction_agreement\':float(agreement.mean()),\'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'reference_complete_coverage\':float((truth!=0).all(axis=1).mean()),\'resample_complete_coverage\':float((pred!=0).all(axis=1).mean()),\n        \'unresolved_learner_rate\':float((pred==0).any(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(paired.mean()),\n        \'n_resolved_profiles\':int(current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique()),\n        \'ari\':float(adjusted_rand_score(ref.cluster_id,current.cluster_id)),\n        \'nmi\':float(normalized_mutual_info_score(ref.cluster_id,current.cluster_id)),\n        \'ami\':float(adjusted_mutual_info_score(ref.cluster_id,current.cluster_id))}\n    for di,keydim in enumerate(KEYS):\n        row[keydim+\'_agreement\']=float(agreement[:,di].mean())\n        row[keydim+\'_joint_resolved_coverage\']=float(paired[:,di].mean())\n        row[keydim+\'_resolved_only_agreement\']=float(agreement[:,di].sum()/paired[:,di].sum()) if paired[:,di].sum() else np.nan\n    refscore=ref[[k+\'_score\' for k in KEYS]].to_numpy(float);score=current[[k+\'_score\' for k in KEYS]].to_numpy(float)\n    row[\'mean_absolute_signed_score_change\']=float(np.nanmean(np.abs(score-refscore))) if paired.any() else np.nan\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':context[\'ids\'],\'source_row_index\':idx,\'reference_cluster_id\':ref.cluster_id,\n                            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\n                            \'all_four_agreement\':complete})\n        for di,k in enumerate(KEYS):\n            frame[k+\'_reference\']=ref[k];frame[k+\'_resample\']=current[k];frame[k+\'_agreement\']=agreement[:,di]\n        member_path=output/\'LMS_STABILITAS/ANGGOTA\'/(\'rep_%02d_\' % rep+re.sub(r\'[^A-Za-z0-9]+\',\'_\',method.replace(\'++\',\'_PP\'))+\'.csv.gz\')\n        save_csv(member_path,frame);paths.append(member_path)\n    cache_write(files,path,signature,paths,{\'summary\':clean(row),\'identity\':identity})\n    return row,\'CALCULATE\'\n\n\ndef summarize_stability(rows):\n    raw=pd.DataFrame(rows);summary=[]\n    metrics=[\'mean_direction_agreement\',\'all_four_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\n             \'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\',\'ari\',\'nmi\',\'ami\']+[k+\'_agreement\' for k in KEYS]\n    for method in ORDER:\n        frame=raw.loc[raw.method.eq(method)]\n        if len(frame)!=30 or frame.rep.nunique()!=30: raise RuntimeError(\'Stabilitas harus berisi 30 resample per metode.\')\n        row={\'method\':method,\'n_resamples\':30}\n        for metric in metrics:\n            row[metric+\'_mean\']=frame[metric].mean();row[metric+\'_sd\']=frame[metric].std(ddof=1)\n            row[metric+\'_min\']=frame[metric].min();row[metric+\'_max\']=frame[metric].max()\n        summary.append(row)\n    paired=[];care=raw.loc[raw.method.eq(\'CARE-KM\')].set_index(\'rep\')\n    for method in [m for m in ORDER if m!=\'CARE-KM\']:\n        other=raw.loc[raw.method.eq(method)].set_index(\'rep\').loc[care.index]\n        for metric in [\'mean_direction_agreement\',\'base_ls_agreement\',\'ari\']:\n            delta=care[metric]-other[metric]\n            paired.append({\'baseline\':method,\'metric\':metric,\'care_minus_baseline_mean\':delta.mean(),\'sd_matched_differences\':delta.std(ddof=1),\n                           \'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                           \'interpretation\':\'descriptive matched overlapping resamples; not 30 independent cohorts\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(paired)\n\n\ndef figures(output,ils,stability):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    paths=[];folder=output/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True)\n    data=ils.set_index(\'method\').loc[ORDER];y=np.arange(len(data))\n    fig,ax=plt.subplots(figsize=(8.5,5))\n    values=data.balanced_accuracy_mean.to_numpy()*100\n    # Percentile CIs need not contain the point estimate; draw their actual endpoints.\n    low=data.balanced_accuracy_ci95_low.to_numpy()*100;high=data.balanced_accuracy_ci95_high.to_numpy()*100\n    ax.hlines(y,low,high,color=\'#333333\');ax.plot(values,y,\'o\',color=\'#185785\')\n    ax.set_yticks(y,ORDER);ax.invert_yaxis();ax.set_xlabel(\'Mean balanced accuracy (%)\');ax.set_xlim(0,100)\n    ax.set_title(\'Pendikom holdout: direct profiles and paired learner bootstrap CIs\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_ils_mean_balanced_accuracy.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5));groups=[stability.loc[stability.method.eq(m),\'mean_direction_agreement\'].to_numpy()*100 for m in ORDER]\n    ax.boxplot(groups,positions=np.arange(1,9),widths=.55)\n    ax.set_xticks(np.arange(1,9),ORDER,rotation=30,ha=\'right\');ax.set_ylim(0,100)\n    ax.set_ylabel(\'Mean agreement of four directions (%)\');ax.set_title(\'LMS semantic stability across 30 matched 80% resamples\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_lms_semantic_stability.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    return paths\n\n\ndef unchanged(files,watched,code_checks):\n    for path,checksum in watched.items():\n        if files.hash(path)!=checksum: raise RuntimeError(\'Masukan berubah selama eksekusi: \'+path)\n    for stem,tag,var,expected in code_checks:\n        if notebook_hash(files,stem,tag,var)!=expected: raise RuntimeError(\'Kode notebook berubah selama eksekusi: \'+stem)\n\n\ndef run_semantics(root,configuration,embedded_source):\n    root=Path(root).expanduser().resolve();files=Files(root);cfg=check_config(configuration)\n    protocol,rules,watched=input_setup(files)\n    own_hash=notebook_hash(files,STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\')\n    code_checks=[(UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',protocol[\'code\'][\'notebook_code\']),\n                 (STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',own_hash)]\n    code_state={\'implementation_sha256\':sha(embedded_source.encode(\'utf-8\')),\'notebook_code\':own_hash,\'runtime\':runtime()}\n    output=root/\'OUTPUT/TAMBAHAN\'/STAGE;control=output/\'KONTROL\'\n    output.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code_state,\'upstream_protocol\':protocol,\n        \'semantic_rule\':\'mode-first; member tie-break; unchanged member-derived 11-unit magnitude\',\n        \'primary_metric\':\'Pendikom mean balanced accuracy\',\'stability_reference\':\'own initializer full run 0; no reference chosen by outcomes\',\n        \'method_count\':8,\'mapping_jobs_expected\':432,\'stability_jobs_expected\':240,\'bootstrap_unit\':\'learner; all methods/runs/dimensions paired\',\n        \'unknown_vs_balanced\':\'UNRESOLVED direction differs from Balanced preference magnitude {1,3}\',\n        \'no_new_clustering\':True})\n    for ds in [\'LMS\',\'PENDIKOM\']: save_csv(output/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),rules[ds])\n    counts={\'CACHE\':0,\'MAP\':0};stability_counts={\'CACHE\':0,\'CALCULATE\':0};rows=[];artifacts=[];stable=[];references={}\n    predictions={m:[] for m in METHODS};reference=None;truth=None;all_watched=dict(watched)\n    try:\n        for ci,context in enumerate(contexts(files),1):\n            unchanged(files,watched,code_checks)\n            print(f"[{ci}/33] {context[\'scope\']} | rep={context[\'rep\']} | n={len(context[\'X\'])}",flush=True)\n            for method in METHODS:\n                runs=15 if method in STOCHASTIC and context[\'scope\']!=\'LMS_RESAMPLING\' else 1\n                for run in range(runs):\n                    upstream=upstream_job(files,protocol,context,method,run);all_watched.update(upstream[\'watched\'])\n                    mapping,evidence,record,summary,disposition=semantic_job(files,output,control,context,upstream,rules[context[\'dataset\']],code_state,cfg)\n                    counts[disposition]+=1;rows.append(summary)\n                    artifacts.append({**upstream[\'identity\'],\'upstream_record_sha256\':upstream[\'checksum\'],\n                                      \'files\':json.dumps(list(record[\'outputs\']),ensure_ascii=False),\n                                      \'upstream_files\':json.dumps(list(upstream[\'record\'][\'outputs\']),ensure_ascii=False)})\n                    if context[\'scope\']==\'LMS_FULL\' and run==0:\n                        frame=profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])\n                        references[method]={\'profiles\':frame,\'signature\':digest({\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\')})}\n                    if context[\'scope\']==\'LMS_RESAMPLING\':\n                        row,dispo=stability_job(files,output,control,context,upstream,mapping,references[method],code_state,cfg)\n                        stable.append(row);stability_counts[dispo]+=1\n                    if context[\'scope\']==\'PENDIKOM_TRAIN\':\n                        if reference is None: reference,truth=ils_reference(files,context)\n                        predictions[method].append(numeric_profiles(profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\'])))\n            save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_mapping_jobs\':len(rows),\'mapping_counts\':counts,\'last_context\':context[\'scope\']})\n        if len(rows)!=432 or len(stable)!=240: raise RuntimeError(\'Jumlah partisi/stabilitas belum lengkap.\')\n        unchanged(files,all_watched,code_checks)\n        ils,ils_cached=evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg)\n        raw,stability,paired=summarize_stability(stable)\n        save_csv(output/\'01_ringkasan_semua_partisi.csv\',pd.DataFrame(rows))\n        save_csv(output/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        save_csv(output/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw)\n        save_csv(output/\'LMS_STABILITAS/02_ringkasan_8_initializer.csv\',stability)\n        save_csv(output/\'LMS_STABILITAS/03_perbedaan_berpasangan_deskriptif.csv\',paired)\n        save_json(output/\'LMS_STABILITAS/04_protokol_stabilitas.json\',{\n            \'reference\':\'own-method LMS full run 0 (seed 4161600), declared in stage 25\',\n            \'resampling\':\'30 fixed matched 80% samples; overlapping learners\',\n            \'inference\':\'descriptive mean/SD/range and matched differences; no independent-cohort p-values\',\n            \'unresolved\':\'never counted as correct direction agreement; conditional agreement includes paired coverage\',\n            \'assignment_source\':\'stage 25 bank; no reclustering\', \'compressed_members\':\'CSV inside gzip; learner IDs, profiles and agreements are auditable\'})\n        summary=pd.DataFrame(rows)\n        save_csv(output/\'LMS_FULL/01_keragaman_profil_per_run.csv\',summary.loc[summary.scope.eq(\'LMS_FULL\')])\n        paper=ils.set_index(\'method\').loc[ORDER].reset_index()\n        for metric in METRIC_NAMES:\n            paper[metric+\'_mean_ci95\']=paper.apply(lambda r: f"{100*r[metric+\'_mean\']:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        save_csv(output/\'TABEL/01_initializer_ils.csv\',paper[[\'method\',\'n_runs\']+[m+\'_mean_ci95\' for m in METRIC_NAMES]+[\'coverage_mean\',\'complete_profile_coverage\']])\n        save_csv(output/\'TABEL/02_initializer_stabilitas.csv\',stability[[\'method\',\'n_resamples\',\'mean_direction_agreement_mean\',\'mean_direction_agreement_sd\',\n            \'base_ls_agreement_mean\',\'base_ls_agreement_sd\',\'unresolved_learner_rate_mean\',\'n_resolved_profiles_mean\',\'ari_mean\']])\n        # Render only when figure inputs/code changed or a figure is missing/tampered.\n        fig_signature=digest({\'code\':code_state,\'ils\':sha((output/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\n                              \'stability\':sha((output/\'LMS_STABILITAS/01_metrik_per_resample.csv\').read_bytes())})\n        fig_cache=cache_read(files,control/\'gambar.json\',fig_signature,cfg[\'force\'])\n        if not fig_cache: cache_write(files,control/\'gambar.json\',fig_signature,figures(output,ils,raw),{})\n        unchanged(files,all_watched,code_checks)\n        save_json(output/\'PROTOKOL/03_manifest_masukan.json\',all_watched)\n        status={\'status\':\'selesai\',\'completed_mapping_jobs\':432,\'completed_stability_jobs\':240,\'method_count\':8,\n                \'mapping_counts\':counts,\'stability_counts\':stability_counts,\'ils_metrics_cached\':ils_cached,\n                \'full_protocol\':cfg[\'bootstrap_repetitions\']>=10000,\'n_ils_test\':139,\n                \'new_clustering_runs\':0,\'output\':str(output),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        # Manifest includes stage-26 scientific deliverables, not mutable control logs.\n        outputs={str(p.relative_to(root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in output.rglob(\'*\') if p.is_file() and control not in p.parents}\n        save_json(control/\'manifest_keluaran.json\',outputs);save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 26 selesai. Partisi:\',counts,\'| Stabilitas:\',stability_counts,\'| ILS cache:\',ils_cached,flush=True)\n        print(\'Hasil:\',output)\n        return {\'ils\':ils,\'stability\':stability,\'status\':status,\'output\':str(output)}\n    except BaseException as error:\n        save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'completed_mapping_jobs\':len(rows),\n                  \'mapping_counts\':counts,\'error\':type(error).__name__+\': \'+str(error),\n                  \'resume\':\'jalankan notebook 26 kembali; cache diperiksa berdasarkan isi\'})\n        raise\n'

_s26 = types.ModuleType('_stage26_helpers')
exec(compile(HELPER26_SOURCE, '_stage26_helpers', 'exec'), _s26.__dict__)
_s26.EMBEDDED_SOURCE = HELPER26_SOURCE


In [4]:
SEMANTIC_K_SOURCE = 'def interpret(X,modes,labels,features,rules,k):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    if modes.shape!=(k,len(features)) or not np.isin(labels,np.arange(k)).all():\n        raise ValueError(\'Ukuran mode/label tidak sesuai k yang diminta.\')\n    sizes=np.bincount(labels,minlength=k);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(k):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n'

_semantic_k = types.ModuleType('_semantic_k29')
for _name in ['np', 'pd', 'DIMS', 'POLES', 'KEYS', 'ABBR', 'LS_LOOKUP']:
    _semantic_k.__dict__[_name] = getattr(_s26, _name)
exec(compile(SEMANTIC_K_SOURCE, '_semantic_k29', 'exec'), _semantic_k.__dict__)

K29_SOURCE = '"""Predeclared cluster-cardinality sensitivity; q=16 and primary semantics fixed."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport io\nimport json\nimport time\nimport types\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'29_Sensitivitas_Jumlah_Cluster_K\'\nGRID=[8,12,16,20,24]\nU26=\'OUTPUT/TAMBAHAN/26_Pengaruh_Initializer_Profil_Semantik\'\nU27=\'OUTPUT/TAMBAHAN/27_Perbandingan_Pembentukan_Profil\'\nDEFAULT_CONFIG={\'k_values\':GRID,\'resample_runs\':30,\'reuse_k16_if_compatible\':True,\'save_member_comparisons\':True,\'force\':False}\n\n\ndef validate_config(configuration):\n    if set(configuration)-set(DEFAULT_CONFIG):raise ValueError(\'Konfigurasi tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration);cfg[\'k_values\']=list(cfg[\'k_values\'])\n    if (not cfg[\'k_values\'] or any(isinstance(k,bool) or not isinstance(k,int) for k in cfg[\'k_values\'])\n        or len(set(cfg[\'k_values\']))!=len(cfg[\'k_values\']) or set(cfg[\'k_values\'])-set(GRID) or 16 not in cfg[\'k_values\']):\n        raise ValueError(\'Gunakan subset grid 8,12,16,20,24 yang memuat reference 16; default seluruh grid.\')\n    cfg[\'k_values\']=sorted(cfg[\'k_values\'])\n    if isinstance(cfg[\'resample_runs\'],bool) or not isinstance(cfg[\'resample_runs\'],int) or not 1<=cfg[\'resample_runs\']<=30:\n        raise ValueError(\'resample_runs harus 1–30; default publikasi 30.\')\n    for key in [\'reuse_k16_if_compatible\',\'save_member_comparisons\',\'force\']:\n        if not isinstance(cfg[key],bool):raise ValueError(key+\' harus True/False.\')\n    return cfg\n\n\ndef runtime_compatible(current,source):\n    return current.get(\'python\')==source.get(\'python\') and all(current[\'packages\'].get(k)==source[\'packages\'].get(k) for k in [\'numpy\',\'pandas\',\'scipy\',\'sklearn\'])\n\n\ndef prepare(s,files):\n    p25,rules,watched=s.input_setup(files)\n    p26=json.loads(files.read(U26+\'/PROTOKOL/01_konfigurasi.json\'))\n    if p26[\'upstream_protocol\']!=p25 or p26[\'code\'][\'implementation_sha256\']!=s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')):\n        raise RuntimeError(\'Sumber tahap 26 tidak sesuai helper yang dibekukan.\')\n    checks=[(s.UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',p25[\'code\'][\'notebook_code\']),\n        (s.STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',p26[\'code\'][\'notebook_code\'])]\n    p27=json.loads(files.read(U27+\'/PROTOKOL/01_konfigurasi.json\'));status=json.loads(files.read(U27+\'/KONTROL/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'full_protocol\') is not True:raise RuntimeError(\'Tahap 27 harus selesai sebelum bobot bootstrap digunakan.\')\n    if p27[\'code\'][\'helper26_sha256\']!=p26[\'code\'][\'implementation_sha256\']:raise RuntimeError(\'Sumber helper tahap 27 tidak sesuai.\')\n    stem=\'27_Perbandingan_Pembentukan_Profil\';checks.append((stem,\'profile27_configuration\',\'KONFIGURASI_27\',p27[\'code\'][\'notebook_code\']))\n    nb=json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\'));literals={}\n    for c in nb[\'cells\']:\n        if c[\'cell_type\']==\'code\':\n            for node in ast.parse(\'\'.join(c[\'source\'])).body:\n                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):literals[node.targets[0].id]=node.value.value\n    if s.sha(literals[\'PROFILE27_SOURCE\'].encode(\'utf-8\'))!=p27[\'code\'][\'implementation_sha256\']:raise RuntimeError(\'Implementasi sumber 27 berbeda.\')\n    for stem,tag,var,h in checks:\n        if s.notebook_hash(files,stem,tag,var)!=h:raise RuntimeError(\'Kode sumber berubah: \'+stem)\n    record_path=U27+\'/KONTROL/evaluasi_ils.json\';saved=s.read_checkpoint(files.path(record_path))\n    if not saved.get(\'signature\'):raise RuntimeError(\'Catatan evaluasi sumber 27 belum tersedia.\')\n    record=s.cache_read(files,files.path(record_path),saved[\'signature\'])\n    if record is None:raise RuntimeError(\'Artefak evaluasi/bootstrap tahap 27 berubah; pulihkan atau jalankan tahap sumber.\')\n    te=files.frame(\'OUTPUT/SPLIT/PENDIKOM/TEST/categorical.csv\',True)\n    reference,truth=s.ils_reference(files,{\'test_frame\':te,\'test_ids\':te.userid.tolist()})\n    wp=U27+\'/ILS/09_bootstrap_weights.npz\'\n    with np.load(io.BytesIO(files.read(wp)),allow_pickle=False) as z:\n        weights=z[\'weights\'].copy();valid=z[\'valid_draws\'].copy();ids=z[\'learner_ids\'].copy()\n    if (weights.ndim!=2 or weights.shape[1]!=139 or len(weights)<10000 or not np.issubdtype(weights.dtype,np.integer)\n        or not (weights.sum(axis=1)==139).all() or len(valid)!=len(weights) or not np.array_equal(ids,reference.userid.to_numpy(str))):\n        raise RuntimeError(\'Bobot bootstrap tidak sejajar dengan 139 peserta test.\')\n    pos=weights.astype(float)@(truth==1)\n    if not np.array_equal(valid,((pos>0)&(pos<139)).all(axis=1)) or valid.sum()<10000:raise RuntimeError(\'Valid draws bootstrap tidak sesuai.\')\n    watched.update(record[\'outputs\']);watched[record_path]=files.hash(record_path)\n    for p in [U26+\'/PROTOKOL/01_konfigurasi.json\',U27+\'/PROTOKOL/01_konfigurasi.json\']:watched[p]=files.hash(p)\n    module=types.ModuleType(\'_care29\');module.__file__=str(files.path(\'LIB/care_km.py\'))\n    exec(compile(files.read(\'LIB/care_km.py\'),module.__file__,\'exec\'),module.__dict__)\n    if module.derive_refinement_budget(16)!=4:raise RuntimeError(\'Batas refinement q=16 harus 4.\')\n    return {\'p25\':p25,\'p26\':p26,\'rules\':rules,\'watched\':watched,\'checks\':checks,\'core\':module,\n        \'reference\':reference,\'truth\':truth,\'weights\':weights,\'valid_weights\':valid,\'weight_path\':wp}\n\n\ndef source_reference(s,files,data,ctx):\n    up=s.upstream_job(files,data[\'p25\'],ctx,\'CARE-KM\',0)\n    rules=data[\'rules\'][ctx[\'dataset\']]\n    sig=s.digest({\'code\':data[\'p26\'][\'code\'],\'upstream_record_sha256\':up[\'checksum\'],\n        \'matrix_sha256\':ctx[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    path=U26+\'/KONTROL/PROFIL/\'+s.digest(up[\'identity\'])[:24]+\'.json\';record=s.cache_read(files,files.path(path),sig)\n    if record is None or record.get(\'upstream_artifacts\')!=up[\'watched\']:raise RuntimeError(\'Artefak profil reference k=16 berubah/tidak sesuai.\')\n    def find(suffix):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+suffix)]\n        if len(found)!=1:raise RuntimeError(\'Artefak reference k=16 tidak lengkap: \'+suffix)\n        return found[0]\n    mapping=files.frame(find(\'01_cluster_profiles.csv\'));evidence=files.frame(find(\'02_cluster_dimension_evidence.csv\'))\n    if mapping.cluster_id.tolist()!=list(range(16)) or not np.array_equal(mapping.cluster_size,np.bincount(up[\'labels\'],minlength=16)):\n        raise RuntimeError(\'Mapping reference k=16 tidak sejajar.\')\n    watched={**up[\'watched\'],path:files.hash(path),**record[\'outputs\']}\n    fit={\'modes\':up[\'modes\'],\'labels\':up[\'labels\'],\'test_labels\':up[\'test_labels\'],\'metrics\':up[\'record\'][\'row\'],\n        \'valid\':True,\'flags\':[],\'origin\':\'REUSED_25_26\',\'signature\':up[\'record\'][\'signature\'],\'outputs\':up[\'record\'][\'outputs\']}\n    profile={\'mapping\':mapping,\'evidence\':evidence,\'signature\':sig,\'outputs\':record[\'outputs\'],\n        \'mapping_file\':find(\'01_cluster_profiles.csv\'),\'evidence_file\':find(\'02_cluster_dimension_evidence.csv\')}\n    return fit,profile,up,watched\n\n\ndef identity(ctx,k):return {\'scope\':ctx[\'scope\'],\'dataset\':ctx[\'dataset\'],\'rep\':ctx[\'rep\'],\'k\':k}\n\n\ndef location(out,kind,ctx,k):\n    return out/kind/(\'k_%02d\'%k)/ctx[\'scope\']/(\'r%02d\'%ctx[\'rep\'] if ctx[\'rep\'] is not None else \'full\')\n\n\ndef load_fit(s,files,record,ctx,k):\n    if not record[\'fit_valid\']:return {\'valid\':False,\'flags\':record[\'flags\'],\'metrics\':record[\'metrics\'],\'origin\':record[\'origin\'],\n        \'signature\':record[\'signature\'],\'outputs\':record[\'outputs\']}\n    def find(name):return next(p for p in record[\'outputs\'] if p.endswith(\'/\'+name))\n    m=files.frame(find(\'02_final_modes.csv\'),True);a=files.frame(find(\'03_assignments.csv.gz\'),True)\n    if m.columns.tolist()!=[\'cluster_id\']+ctx[\'features\'] or m.cluster_id.astype(int).tolist()!=list(range(k)) or a.identity.tolist()!=ctx[\'ids\']:\n        raise RuntimeError(\'Schema cache partisi tidak sesuai.\')\n    modes=m[ctx[\'features\']].to_numpy(object);labels=a.cluster_id.to_numpy(int)\n    if not np.array_equal(a.source_row_index.astype(np.int64),ctx[\'indices\']) or not np.isin(labels,np.arange(k)).all():raise RuntimeError(\'Indeks cache partisi tidak sesuai.\')\n    test=None\n    if ctx[\'test_X\'] is not None:\n        f=files.frame(find(\'04_test_assignments.csv\'),True)\n        if f.identity.tolist()!=ctx[\'test_ids\']:raise RuntimeError(\'ID test pada cache tidak sesuai.\')\n        test=f.cluster_id.to_numpy(int)\n    return {\'valid\':True,\'modes\':modes,\'labels\':labels,\'test_labels\':test,\'metrics\':record[\'metrics\'],\'flags\':record[\'flags\'],\n        \'origin\':record[\'origin\'],\'signature\':record[\'signature\'],\'outputs\':record[\'outputs\']}\n\n\ndef fit_partition(s,files,out,control,ctx,k,data,code,cfg):\n    params={\'identity\':identity(ctx,k),\'matrix_sha256\':ctx[\'matrix_sha256\'],\'q\':16,\'derived_refinement_budget\':4,\n        \'care_config\':data[\'p25\'][\'constants\'][\'FROZEN_CARE_CONFIG\'],\'max_iter\':100,\n        \'si_sample_size\':None if ctx[\'scope\']==\'PENDIKOM_TRAIN\' else 2000,\n        \'si_seed\':6161600+ctx[\'rep\'] if ctx[\'rep\'] is not None else 4161600}\n    sig=s.digest({\'code\':code,\'parameters\':params});cp=control/\'PARTISI\'/(s.digest(identity(ctx,k))[:24]+\'.json\')\n    record=s.cache_read(files,cp,sig,cfg[\'force\'])\n    if record:return load_fit(s,files,record,ctx,k),\'CACHE\'\n    X=ctx[\'X\'];core=data[\'core\'];flags=[];paths=[];folder=location(out,\'PARTISI\',ctx,k)\n    if len({tuple(row) for row in X})<k:raise ValueError(\'k melebihi jumlah vektor unik pada \'+ctx[\'scope\'])\n    start=time.perf_counter();valid=False;metrics={}\n    try:\n        init,trace=core.care_init(X,k,return_trace=True,**params[\'care_config\']);t1=time.perf_counter()\n        fit=core.kmodes_fit(X,init,max_iter=100);t2=time.perf_counter()\n        labels=np.asarray(fit[\'labels\'],int);modes=np.asarray(fit[\'modes\'],object)\n        if init.shape!=(k,len(ctx[\'features\'])) or modes.shape!=init.shape or len(labels)!=len(X) or not np.isin(labels,np.arange(k)).all():\n            raise RuntimeError(\'Ukuran mode/label hasil fitting tidak sesuai k.\')\n        counts=np.bincount(labels,minlength=k)\n        initial=float(core.initial_cost(X,init));final=float(fit[\'cost\'])\n        if (counts==0).any():flags.append(\'empty_final_cluster\')\n        if final>initial+1e-12:flags.append(\'objective_worsened\')\n        if int(fit[\'iterations\'])<1:flags.append(\'nonpositive_iterations\')\n        valid=not flags\n        metrics={\'initial_cost\':initial,\'final_cost\':final,\'iterations\':int(fit[\'iterations\']),\n            \'min_cluster\':int(counts.min()),\'max_cluster\':int(counts.max()),\'median_cluster\':float(np.median(counts)),\n            \'iteration_cap_reached\':int(fit[\'iterations\'])>=100,\'accepted_refinements\':len(trace),\n            \'init_sec\':t1-start,\'cluster_sec\':t2-t1,\'dbi\':None,\'si\':None}\n        if valid:\n            try:\n                mm=core.evaluate_clustering(X,None,fit,init,si_sample_size=params[\'si_sample_size\'],si_seed=params[\'si_seed\'])\n                metrics.update(mm)\n                if not all(np.isfinite(float(mm[x])) for x in [\'dbi\',\'si\']):metrics[\'internal_metric_status\']=\'nonfinite\'\n                else:metrics[\'internal_metric_status\']=\'valid\'\n            except (ValueError,FloatingPointError) as error:metrics[\'internal_metric_status\']=\'undefined: \'+str(error)\n        for name,array in [(\'01_initial_modes.csv\',init),(\'02_final_modes.csv\',modes)]:\n            frame=pd.DataFrame(array,columns=ctx[\'features\']);frame.insert(0,\'cluster_id\',np.arange(k));p=folder/name;s.save_csv(p,frame);paths.append(p)\n        p=folder/\'03_assignments.csv.gz\';s.save_csv(p,pd.DataFrame({\'identity\':ctx[\'ids\'],\'source_row_index\':ctx[\'indices\'],\'cluster_id\':labels}));paths.append(p)\n        if ctx[\'test_X\'] is not None:\n            test,distance=core.assign_nearest(ctx[\'test_X\'],modes);p=folder/\'04_test_assignments.csv\'\n            s.save_csv(p,pd.DataFrame({\'identity\':ctx[\'test_ids\'],\'cluster_id\':test,\'distance\':distance}));paths.append(p)\n        p=folder/\'05_refinement_trace.json\';s.save_json(p,trace);paths.append(p)\n    except (ValueError,RuntimeError,FloatingPointError) as error:\n        flags.append(type(error).__name__+\': \'+str(error));valid=False\n        p=folder/\'fit_error.json\';s.save_json(p,{\'error\':str(error),\'type\':type(error).__name__});paths.append(p)\n    record=s.cache_write(files,cp,sig,paths,{\'identity\':identity(ctx,k),\'parameters\':params,\'fit_valid\':valid,\n        \'flags\':flags,\'metrics\':s.clean(metrics),\'origin\':\'COMPUTED_CURRENT_ENVIRONMENT\'})\n    return load_fit(s,files,record,ctx,k),\'FIT\'\n\n\ndef profile_partition(s,files,out,control,ctx,k,fit,rules,code,cfg,semantic):\n    sig=s.digest({\'code\':code,\'fit_signature\':fit[\'signature\'],\'fit_outputs\':fit[\'outputs\'],\'rules\':rules.to_dict(\'records\'),\'k\':k})\n    cp=control/\'PROFIL\'/(s.digest(identity(ctx,k))[:24]+\'.json\');record=s.cache_read(files,cp,sig,cfg[\'force\'])\n    folder=location(out,\'PROFIL\',ctx,k);mp=folder/\'01_cluster_mapping.csv\';ep=folder/\'02_dimension_evidence.csv\'\n    if record:\n        mapping=files.frame(str(mp.relative_to(files.root)).replace(\'\\\\\',\'/\'));evidence=files.frame(str(ep.relative_to(files.root)).replace(\'\\\\\',\'/\'));dispo=\'CACHE\'\n    else:\n        mapping,evidence=semantic.interpret(ctx[\'X\'],fit[\'modes\'],fit[\'labels\'],ctx[\'features\'],rules,k)\n        s.save_csv(mp,mapping);s.save_csv(ep,evidence);paths=[mp,ep]\n        if ctx[\'test_X\'] is not None:\n            for name,frame in [(\'03_train_targets.csv.gz\',s.profiles(mapping,fit[\'labels\'],ctx[\'ids\'],ctx[\'indices\'])),\n                (\'04_test_profiles.csv\',s.profiles(mapping,fit[\'test_labels\'],ctx[\'test_ids\']))]:\n                p=folder/name;s.save_csv(p,frame);paths.append(p)\n        record=s.cache_write(files,cp,sig,paths,{\'identity\':identity(ctx,k)});dispo=\'MAP\'\n    if mapping.cluster_id.tolist()!=list(range(k)) or len(evidence)!=4*k:raise RuntimeError(\'Jumlah baris profil tidak sesuai k.\')\n    return {\'mapping\':mapping,\'evidence\':evidence,\'signature\':sig,\'outputs\':record[\'outputs\'],\n        \'mapping_file\':str(mp.relative_to(files.root)).replace(\'\\\\\',\'/\'),\'evidence_file\':str(ep.relative_to(files.root)).replace(\'\\\\\',\'/\')},dispo\n\n\ndef profile_summary(s,ctx,k,fit,profile):\n    row={**identity(ctx,k),\'n_learners\':len(ctx[\'X\']),\'fit_valid\':fit[\'valid\'],\'fit_flags\':\'; \'.join(fit[\'flags\']),\n        \'fit_origin\':fit[\'origin\'],**fit[\'metrics\']}\n    if not fit[\'valid\']:return row\n    m=profile[\'mapping\'];e=profile[\'evidence\'];n=len(ctx[\'X\']);resolved=m.mapping_status.eq(\'RESOLVED\')\n    dist=m.loc[resolved].groupby(\'ls_class\').cluster_size.sum();probs=dist/dist.sum() if dist.sum() else pd.Series(dtype=float)\n    if len(dist)>min(k,16):raise RuntimeError(\'Jumlah Base LS melebihi kapasitas k atau taxonomy 16.\')\n    unresolved=e.decision_source.eq(\'BALANCED\')\n    row.update(n_resolved_base_ls=len(dist),unresolved_clusters=int((~resolved).sum()),unresolved_cluster_rate=float((~resolved).mean()),\n        unresolved_cluster_dimensions=int(unresolved.sum()),unresolved_cluster_dimension_rate=float(unresolved.mean()),\n        unresolved_learner_rate=float(m.loc[~resolved,\'cluster_size\'].sum()/n),\n        unresolved_learner_dimension_rate=float(e.loc[unresolved,\'cluster_size\'].sum()/(4*n)),\n        largest_resolved_profile_share_all=float(dist.max()/n) if len(dist) else 0.,\n        resolved_profile_entropy_bits=float(-(probs*np.log2(probs)).sum()),\n        mode_member_conflicts=int((~e.mode_member_consistency.astype(bool)).sum()),\n        strong_mode_member_conflicts=int(e.strong_mode_member_conflict.astype(bool).sum()),mapping_file=profile[\'mapping_file\'])\n    for key in s.KEYS:row[key+\'_unresolved_learner_rate\']=float(m.loc[m[key].eq(\'Balanced\'),\'cluster_size\'].sum()/n)\n    return row\n\n\ndef compare_profiles(s,reference,current):\n    if reference.identity.tolist()!=current.identity.tolist():raise RuntimeError(\'Identitas profil tidak sejajar.\')\n    a=s.numeric_profiles(reference);b=s.numeric_profiles(current);joint=(a!=0)&(b!=0);agree=joint&(a==b)\n    row={\'mean_direction_agreement\':float(agree.mean()),\'all_four_direction_agreement\':float(agree.all(axis=1).mean()),\n        \'base_ls_agreement\':float(agree.all(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(joint.mean()),\n        \'current_unresolved_learner_rate\':float((b==0).any(axis=1).mean()),\n        \'resolved_only_direction_agreement\':float(agree.sum()/joint.sum()) if joint.sum() else np.nan}\n    for di,key in enumerate(s.KEYS):\n        row[key+\'_agreement\']=float(agree[:,di].mean());row[key+\'_joint_resolved_coverage\']=float(joint[:,di].mean())\n        row[key+\'_resolved_only_agreement\']=float(agree[:,di].sum()/joint[:,di].sum()) if joint[:,di].sum() else np.nan\n    return row,agree\n\n\ndef reused_stability(s,files,data,ctx,up,profile,reference):\n    cp=U26+\'/KONTROL/STABILITAS/\'+s.digest(up[\'identity\'])[:24]+\'.json\'\n    sig=s.digest({\'code\':data[\'p26\'][\'code\'],\'upstream\':up[\'checksum\'],\'mapping\':profile[\'mapping\'].to_dict(\'records\'),\n        \'reference\':reference[\'source_signature\'],\'save_members\':data[\'p26\'][\'configuration\'][\'save_member_comparisons\']})\n    record=s.cache_read(files,files.path(cp),sig)\n    if record is None:raise RuntimeError(\'Reference stabilitas k=16 berubah/tidak sesuai.\')\n    return {**record[\'summary\'],\'k\':16,\'valid_stability\':True,\'stability_origin\':\'REUSED_26\'}, {cp:files.hash(cp),**record[\'outputs\']}\n\n\ndef stability_job(s,files,out,control,ctx,k,fit,profile,reference,code,cfg):\n    sig=s.digest({\'code\':code,\'k\':k,\'fit_signature\':fit[\'signature\'],\'profile_signature\':profile[\'signature\'],\n        \'reference\':reference[\'signature\'],\'matrix\':ctx[\'matrix_sha256\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    cp=control/\'STABILITAS\'/(s.digest(identity(ctx,k))[:24]+\'.json\');record=s.cache_read(files,cp,sig,cfg[\'force\'])\n    if record:return record[\'summary\'],\'CACHE\'\n    from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score,adjusted_mutual_info_score\n    current=s.profiles(profile[\'mapping\'],fit[\'labels\'],ctx[\'ids\'],ctx[\'indices\'])\n    ref=reference[\'profiles\'].iloc[ctx[\'indices\']].reset_index(drop=True);row,agree=compare_profiles(s,ref,current)\n    row.update(**identity(ctx,k),n_learners=len(current),valid_stability=True,stability_origin=\'COMPUTED\',\n        unresolved_learner_rate=row[\'current_unresolved_learner_rate\'],\n        n_resolved_profiles=current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique(),\n        ari=adjusted_rand_score(ref.cluster_id,current.cluster_id),nmi=normalized_mutual_info_score(ref.cluster_id,current.cluster_id),\n        ami=adjusted_mutual_info_score(ref.cluster_id,current.cluster_id))\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':ctx[\'ids\'],\'source_row_index\':ctx[\'indices\'],\'reference_cluster_id\':ref.cluster_id,\n            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\n            \'all_four_agreement\':agree.all(axis=1)})\n        for di,key in enumerate(s.KEYS):frame[key+\'_reference\']=ref[key];frame[key+\'_resample\']=current[key];frame[key+\'_agreement\']=agree[:,di]\n        path=out/\'LMS_STABILITAS/ANGGOTA\'/(\'k_%02d_rep_%02d.csv.gz\'%(k,ctx[\'rep\']));s.save_csv(path,frame);paths.append(path)\n    s.cache_write(files,cp,sig,paths,{\'summary\':s.clean(row)});return row,\'CALCULATE\'\n\n\ndef aggregate_stability(s,rows,ks,reps):\n    raw=pd.DataFrame(rows);summary=[];paired=[]\n    metrics=[\'mean_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\',\'ari\',\'nmi\',\'ami\']+[k+\'_agreement\' for k in s.KEYS]\n    for k in ks:\n        group=raw.loc[raw.k.eq(k)];good=group.loc[group.valid_stability.eq(True)]\n        if len(group)!=reps or group.rep.nunique()!=reps:raise RuntimeError(\'Baris stabilitas per k tidak lengkap.\')\n        row={\'k\':k,\'n_planned\':reps,\'n_valid\':len(good),\'n_failed\':reps-len(good)}\n        for metric in metrics:\n            values=pd.to_numeric(good[metric],errors=\'coerce\') if metric in good else pd.Series(dtype=float)\n            row[metric+\'_mean\']=values.mean();row[metric+\'_sd\']=values.std(ddof=1)\n            row[metric+\'_min\']=values.min();row[metric+\'_max\']=values.max()\n        summary.append(row)\n    for k in [v for v in ks if v!=16]:\n        a=raw.loc[raw.k.eq(k)].set_index(\'rep\');b=raw.loc[raw.k.eq(16)].set_index(\'rep\').loc[a.index]\n        valid=a.valid_stability.eq(True)&b.valid_stability.eq(True)\n        for metric in [\'mean_direction_agreement\',\'base_ls_agreement\',\'ari\']:\n            delta=pd.to_numeric(a.loc[valid,metric])-pd.to_numeric(b.loc[valid,metric]) if metric in a and metric in b else pd.Series(dtype=float)\n            paired.append({\'k\':k,\'reference_k\':16,\'metric\':metric,\'k_minus_16_mean\':delta.mean(),\'sd_paired_difference\':delta.std(ddof=1),\n                \'n_valid_pairs\':len(delta),\'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                \'inference\':\'descriptive overlapping resamples; no independent-cohort p-values\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(paired)\n\n\ndef evaluate_ils(s,files,out,control,data,predictions,ks,code,cfg):\n    available=[k for k in ks if predictions.get(k) is not None]\n    sig=s.digest({\'code\':code,\'k_values\':ks,\'truth\':data[\'truth\'].tolist(),\'test_ids\':data[\'reference\'].userid.tolist(),\n        \'predictions\':{str(k):s.sha(np.asarray(predictions[k],np.int8).tobytes()) if k in available else None for k in ks},\n        \'weights_sha256\':files.hash(data[\'weight_path\'])})\n    cp=control/\'evaluasi_ils.json\';folder=out/\'ILS\';record=s.cache_read(files,cp,sig,cfg[\'force\'])\n    if record:return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    truth=data[\'truth\'];weights=data[\'weights\'][data[\'valid_weights\']].astype(float);B=len(weights)\n    boot=np.full((B,len(ks),5,3),np.nan);points={};counts={}\n    print(\'ILS:\',len(available),\'dari\',len(ks),\'nilai k; shared bootstrap:\',B,flush=True)\n    with threadpool_limits(limits=1):\n        for ki,k in enumerate(ks):\n            if k not in available:continue\n            mask=s.confusion_masks(truth,np.asarray(predictions[k],np.int8)[None,:,:])[:,0,:,:]\n            counts[k]=mask.sum(axis=0);points[k]=s.count_metrics(counts[k])\n            metrics=s.count_metrics((weights@mask.reshape(139,-1)).reshape(B,4,6))\n            boot[:,ki,:4]=metrics;boot[:,ki,4]=metrics.mean(axis=1)\n    per_dim=[];overall=[];predrows=[];cm=[];diff=[];refcounts=[]\n    for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n        for value,pole in [(1,poles[0]),(-1,poles[1])]:refcounts.append({\'dimension\':key,\'reference_pole\':pole,\'n_test\':int((truth[:,di]==value).sum())})\n    for ki,k in enumerate(ks):\n        good=k in available\n        if good:\n            arr=np.asarray(predictions[k]);point=points[k]\n            for li,userid in enumerate(data[\'reference\'].userid):\n                row={\'k\':k,\'identity\':userid,\'courseid\':data[\'reference\'].courseid.iloc[li]};combo=[]\n                for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                    value=arr[li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1];row[\'predicted_\'+key]=pole or \'UNRESOLVED\';combo.append(pole)\n                row[\'predicted_ls_class\']=s.LS_LOOKUP[tuple(combo)] if all(combo) else \'AMBIGUOUS\';predrows.append(row)\n            for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                for ci,num in enumerate(counts[k][di]):cm.append({\'k\':k,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n            row={\'k\':k,\'dimension\':key,\'n_test\':139,\'valid_partition\':good,\'status\':\'VALID\' if good else \'FIT_FAILED\',\n                \'n_bootstrap\':B,\'coverage\':float((arr[:,di]!=0).mean()) if good and di<4 else float((arr!=0).mean()) if good else np.nan,\n                \'complete_profile_coverage\':float((arr!=0).all(axis=1).mean()) if good else np.nan}\n            for ji,metric in enumerate(s.METRIC_NAMES):\n                row[metric]=(point[di,ji] if di<4 else point[:,ji].mean()) if good else np.nan\n                row[metric+\'_ci95_low\'],row[metric+\'_ci95_high\']=np.quantile(boot[:,ki,di,ji],[.025,.975]) if good else (np.nan,np.nan)\n            (per_dim if di<4 else overall).append(row)\n    refidx=ks.index(16)\n    for ki,k in enumerate(ks):\n        if k==16:continue\n        good=k in available and 16 in available\n        for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n            for ji,metric in enumerate(s.METRIC_NAMES):\n                row={\'k\':k,\'reference_k\':16,\'dimension\':key,\'metric\':metric,\'valid_pair\':good,\n                    \'primary_family\':di==4 and metric==\'balanced_accuracy\',\'positive_favors\':\'current k; diagnostic, not selection\'}\n                if good:\n                    delta=boot[:,ki,di,ji]-boot[:,refidx,di,ji]\n                    a=points[k][di,ji] if di<4 else points[k][:,ji].mean();b=points[16][di,ji] if di<4 else points[16][:,ji].mean()\n                    row.update(k_minus_16=a-b,paired_ci95_low=np.quantile(delta,.025),paired_ci95_high=np.quantile(delta,.975))\n                    if row[\'primary_family\']:row[\'bonferroni_family95_low\'],row[\'bonferroni_family95_high\']=np.quantile(delta,[.05/8,1-.05/8])\n                diff.append(row)\n    paths=[]\n    for filename,frame in [(\'01_prediksi_per_peserta.csv\',pd.DataFrame(predrows)),(\'02_metrik_per_dimensi.csv\',pd.DataFrame(per_dim)),\n        (\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),(\'05_confusion_matrices.csv\',pd.DataFrame(cm)),\n        (\'07_selisih_berpasangan_vs_k16.csv\',pd.DataFrame(diff)),(\'08_reference_counts.csv\',pd.DataFrame(refcounts))]:\n        p=folder/filename;s.save_csv(p,frame);paths.append(p)\n    buf=io.BytesIO();np.savez_compressed(buf,metrics=boot,k_values=np.asarray(ks),dimensions=np.asarray(s.KEYS+[\'mean_four_dimensions\']),metric_names=np.asarray(s.METRIC_NAMES))\n    p=folder/\'06_bootstrap_paired_metrics.npz\';s.atomic(p,buf.getvalue());paths.append(p)\n    p=folder/\'09_protokol_evaluasi.json\';s.save_json(p,{\'primary_metric\':\'mean balanced accuracy over four dimensions\',\'reference_k\':16,\n        \'predeclared_grid\':GRID,\'k_selection\':\'none; all settings are reported, no test-based argmax\',\n        \'weights_source\':data[\'weight_path\'],\'weights_sha256\':files.hash(data[\'weight_path\']),\'unit\':\'139 paired learners; all four dimensions together\',\n        \'individual_intervals\':\'descriptive percentile 95%\',\'family_intervals\':\'Bonferroni percentile family95 for four prespecified overall mean-BA contrasts against 16\',\n        \'unresolved\':\'included as abstention/error; no test learners dropped\',\'fit_failures\':\'reported separately as unavailable metrics, never relabelled as semantic unresolved\',\n        \'scope\':\'conditional on frozen train split and rules; same cohort, not new independent test cohorts\'})\n    paths.append(p);s.cache_write(files,cp,sig,paths,{\'valid_k\':available,\'k_values\':ks,\'n_learners\':139})\n    return pd.DataFrame(overall),False\n\n\ndef figures(s,out,ils,stability,diversity):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    folder=out/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True);paths=[]\n    fig,ax=plt.subplots(figsize=(8,4.8));good=ils.loc[ils.valid_partition.eq(True)]\n    ax.plot(good.k,100*good.balanced_accuracy,\'o-\',color=\'#185785\');ax.fill_between(good.k,100*good.balanced_accuracy_ci95_low,100*good.balanced_accuracy_ci95_high,alpha=.18,color=\'#185785\')\n    ax.axvline(16,color=\'#777777\',linestyle=\'--\',label=\'Predeclared reference k=16\');ax.set_xticks(ils.k);ax.set_ylim(0,100)\n    ax.set_xlabel(\'Number of clusters k\');ax.set_ylabel(\'Mean balanced accuracy (%)\');ax.set_title(\'Pendikom ILS correspondence: fixed q=16 and primary mapping\');ax.legend(fontsize=10);fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        p=folder/(\'01_k_versus_ILS.\'+ext);fig.savefig(p,dpi=300,bbox_inches=\'tight\');paths.append(p)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(8,4.8));x=stability.k.to_numpy()\n    for metric,label,color in [(\'mean_direction_agreement\',\'Mean four-direction agreement\',\'#185785\'),(\'base_ls_agreement\',\'Complete Base LS agreement\',\'#777777\')]:\n        mean=100*stability[metric+\'_mean\'].to_numpy(float);sd=100*stability[metric+\'_sd\'].to_numpy(float)\n        ax.errorbar(x,mean,yerr=sd,fmt=\'o-\',capsize=3,label=label,color=color)\n    ax.axvline(16,color=\'#999999\',linestyle=\'--\');ax.set_xticks(x);ax.set_ylim(0,100);ax.set_xlabel(\'Number of clusters k\')\n    ax.set_ylabel(\'Agreement (%)\');ax.set_title(\'LMS stability: mean and SD of matched overlapping resamples\');ax.legend(fontsize=10);fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        p=folder/(\'02_k_versus_stabilitas.\'+ext);fig.savefig(p,dpi=300,bbox_inches=\'tight\');paths.append(p)\n    plt.close(fig)\n    return paths\n\n\ndef run_sensitivity(root,configuration,embedded_source,semantic_source,s,semantic):\n    cfg=validate_config(configuration);files=s.Files(Path(root).expanduser().resolve());data=prepare(s,files)\n    own=s.notebook_hash(files,STAGE,\'k29_configuration\',\'KONFIGURASI_29\');checks=data[\'checks\']+[(STAGE,\'k29_configuration\',\'KONFIGURASI_29\',own)]\n    current=s.runtime();can_reuse=cfg[\'reuse_k16_if_compatible\'] and runtime_compatible(current,data[\'p25\'][\'runtime\'])\n    code={\'implementation_sha256\':s.sha(embedded_source.encode(\'utf-8\')),\'semantic_k_sha256\':s.sha(semantic_source.encode(\'utf-8\')),\n        \'helper26_sha256\':s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')),\'notebook_code\':own,\'core_sha256\':files.hash(\'LIB/care_km.py\'),\'runtime\':current}\n    out=files.root/\'OUTPUT/TAMBAHAN\'/STAGE;control=out/\'KONTROL\';out.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    ncontexts=cfg[\'resample_runs\']+2;planned=ncontexts*len(cfg[\'k_values\']);expected_fit=planned-ncontexts if can_reuse else planned\n    s.save_json(out/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code,\'source25_runtime\':data[\'p25\'][\'runtime\'],\n        \'k16_reuse_eligible\':can_reuse,\'runtime_policy\':\'reuse frozen k16 only when scientific package versions and Python match; otherwise refit in current environment\',\n        \'predeclared_grid\':GRID,\'primary_k\':16,\'selection\':\'none; all configured k reported; test not used for choosing k\',\n        \'initializer\':\'CARE-KM\',\'care_config\':data[\'p25\'][\'constants\'][\'FROZEN_CARE_CONFIG\'],\'refinement_bound\':4,\'max_iter\':100,\n        \'primary_semantics\':\'unchanged mode-first, member tie-break, member-derived 11-unit magnitude\',\'Base_LS_space\':16,\n        \'planned_partitions\':planned,\'expected_new_fits_without_cache\':expected_fit,\'LMS_reference\':\'own full-data model for each k\',\n        \'ILS_reference\':\'same 139 test learners, mapping fitted on 322 train learners only\',\n        \'reused_bootstrap\':data[\'weight_path\'],\'SI_sampling\':\'same 2000 positions/seed within each context for all k\',\n        \'failure_policy\':\'algorithmic invalidity distinct from unresolved; no rescue seed or objective/rule change\'})\n    for ds in [\'LMS\',\'PENDIKOM\']:s.save_csv(out/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),data[\'rules\'][ds])\n    print(\'Grid k:\',cfg[\'k_values\'],\'| q=16 | refinement bound=4 | expected partitions:\',planned,flush=True)\n    if can_reuse:print(\'k=16: gunakan artefak terverifikasi tahap 25/26; fitting baru maksimum\',expected_fit,flush=True)\n    else:print(\'k=16 dihitung dalam lingkungan sekarang bersama k lainnya; fitting maksimum\',expected_fit,flush=True)\n    fit_counts={\'REUSED_25_26\':0,\'FIT\':0,\'CACHE\':0};map_counts={\'REUSED_26\':0,\'MAP\':0,\'CACHE\':0};stab_counts={\'REUSED_26\':0,\'CALCULATE\':0,\'CACHE\':0,\'UNAVAILABLE\':0}\n    rows=[];artifacts=[];stable=[];distributions=[];cross=[];references={};predictions={k:None for k in cfg[\'k_values\']};pendframes={}\n    allwatched=dict(data[\'watched\']);completed=0\n    try:\n        for ctx in s.contexts(files):\n            if ctx[\'scope\']==\'LMS_TRAIN\' or (ctx[\'rep\'] is not None and ctx[\'rep\']>=cfg[\'resample_runs\']):continue\n            s.unchanged(files,data[\'watched\'],checks)\n            print(ctx[\'scope\'],\'| rep=\',ctx[\'rep\'],\'| n=\',len(ctx[\'X\']),flush=True)\n            seed=6161600+ctx[\'rep\'] if ctx[\'rep\'] is not None else 4161600\n            idx=(np.sort(np.random.default_rng(seed).choice(len(ctx[\'X\']),size=2000,replace=False)) if len(ctx[\'X\'])>2000 else np.arange(len(ctx[\'X\'])))\n            si_path=out/\'PROTOKOL/SI_INDICES\'/((ctx[\'scope\']+(\'_%02d\'%ctx[\'rep\'] if ctx[\'rep\'] is not None else \'\'))+\'.csv\')\n            s.save_csv(si_path,pd.DataFrame({\'context_row_index\':idx,\'source_row_index\':ctx[\'indices\'][idx],\'identity\':np.asarray(ctx[\'ids\'])[idx]}))\n            for k in cfg[\'k_values\']:\n                if k==16 and can_reuse:\n                    fit,profile,up,ww=source_reference(s,files,data,ctx);allwatched.update(ww)\n                    fit_counts[\'REUSED_25_26\']+=1;map_counts[\'REUSED_26\']+=1\n                else:\n                    fit,dispo=fit_partition(s,files,out,control,ctx,k,data,code,cfg);fit_counts[dispo]+=1;profile=None;up=None\n                    if fit[\'valid\']:\n                        profile,mp=profile_partition(s,files,out,control,ctx,k,fit,data[\'rules\'][ctx[\'dataset\']],code,cfg,semantic);map_counts[mp]+=1\n                completed+=1;summary=profile_summary(s,ctx,k,fit,profile);rows.append(summary)\n                artifacts.append({**identity(ctx,k),\'fit_origin\':fit[\'origin\'],\'fit_valid\':fit[\'valid\'],\n                    \'fit_files\':json.dumps(list(fit[\'outputs\']),ensure_ascii=False),\n                    \'profile_files\':json.dumps(list(profile[\'outputs\']) if profile else [],ensure_ascii=False)})\n                if profile:\n                    m=profile[\'mapping\'];dist=m.groupby(\'ls_class\',dropna=False).cluster_size.sum()\n                    for label,count in dist.items():distributions.append({**identity(ctx,k),\'ls_class\':label,\'n_learners\':int(count),\'share\':count/len(ctx[\'X\'])})\n                if ctx[\'scope\']==\'LMS_FULL\':\n                    if fit[\'valid\']:\n                        ref={\'profiles\':s.profiles(profile[\'mapping\'],fit[\'labels\'],ctx[\'ids\'],ctx[\'indices\']),\n                            \'signature\':s.digest({\'fit\':fit[\'signature\'],\'profile\':profile[\'signature\'],\'outputs\':profile[\'outputs\']})}\n                        if k==16 and can_reuse:ref[\'source_signature\']=s.digest({\'upstream\':up[\'checksum\'],\'mapping\':profile[\'mapping\'].to_dict(\'records\')})\n                        references[k]=ref\n                    else:references[k]=None\n                elif ctx[\'scope\']==\'LMS_RESAMPLING\':\n                    if not fit[\'valid\'] or not references.get(k):\n                        sr={**identity(ctx,k),\'valid_stability\':False,\'reason\':\'invalid resample or full reference\'};sd=\'UNAVAILABLE\'\n                    elif k==16 and can_reuse:\n                        sr,ww=reused_stability(s,files,data,ctx,up,profile,references[k]);allwatched.update(ww);sd=\'REUSED_26\'\n                    else:sr,sd=stability_job(s,files,out,control,ctx,k,fit,profile,references[k],code,cfg)\n                    stable.append(sr);stab_counts[sd]+=1\n                else:\n                    if fit[\'valid\']:\n                        frame=s.profiles(profile[\'mapping\'],fit[\'test_labels\'],ctx[\'test_ids\']);pendframes[k]=frame;predictions[k]=s.numeric_profiles(frame)\n                s.save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_partitions\':completed,\'fit_counts\':fit_counts,\'profile_counts\':map_counts,\'last_k\':k,\'last_scope\':ctx[\'scope\']})\n            if ctx[\'scope\']==\'LMS_FULL\':\n                for k in cfg[\'k_values\']:\n                    if references.get(k) and references.get(16):\n                        cr,_=compare_profiles(s,references[16][\'profiles\'],references[k][\'profiles\']);cross.append({\'cohort\':\'LMS_FULL\',\'k\':k,\'reference_k\':16,\'valid_pair\':True,**cr})\n                    else:cross.append({\'cohort\':\'LMS_FULL\',\'k\':k,\'reference_k\':16,\'valid_pair\':False})\n        if completed!=planned or len(stable)!=len(cfg[\'k_values\'])*cfg[\'resample_runs\']:raise RuntimeError(\'Jumlah konfigurasi belum lengkap.\')\n        for k in cfg[\'k_values\']:\n            if k in pendframes and 16 in pendframes:\n                cr,_=compare_profiles(s,pendframes[16],pendframes[k]);cross.append({\'cohort\':\'PENDIKOM_TEST\',\'k\':k,\'reference_k\':16,\'valid_pair\':True,**cr})\n            else:cross.append({\'cohort\':\'PENDIKOM_TEST\',\'k\':k,\'reference_k\':16,\'valid_pair\':False})\n        s.unchanged(files,allwatched,checks)\n        ils,cached=evaluate_ils(s,files,out,control,data,predictions,cfg[\'k_values\'],code,cfg)\n        raw,stability,paired=aggregate_stability(s,stable,cfg[\'k_values\'],cfg[\'resample_runs\'])\n        summary=pd.DataFrame(rows)\n        s.save_csv(out/\'01_ringkasan_semua_partisi.csv\',summary);s.save_csv(out/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        s.save_csv(out/\'DIVERSITAS/01_distribusi_Base_LS.csv\',pd.DataFrame(distributions))\n        s.save_csv(out/\'CROSS_K/01_agreement_vs_k16.csv\',pd.DataFrame(cross))\n        s.save_csv(out/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw);s.save_csv(out/\'LMS_STABILITAS/02_ringkasan_per_k.csv\',stability)\n        s.save_csv(out/\'LMS_STABILITAS/03_selisih_matched_vs_k16.csv\',paired)\n        s.save_json(out/\'LMS_STABILITAS/04_protokol.json\',{\'within_k\':\'resample vs own full-data reference at same k\',\n            \'cross_k\':\'profile agreement against k16, sensitivity not independent truth accuracy\',\n            \'overlap\':\'30 matched overlapping samples; descriptive mean/SD/range, no independent-cohort p-values\',\n            \'unresolved\':\'never counted as matching direction; conditional agreement accompanied by joint coverage\',\n            \'no_selection\':\'primary k16 retained; no best-k output or test-label tuning\'})\n        failure_columns=[\'scope\',\'dataset\',\'rep\',\'k\',\'fit_valid\',\'fit_flags\',\'fit_origin\',\'iteration_cap_reached\',\'internal_metric_status\']\n        s.save_csv(out/\'AUDIT/01_validitas_partisi.csv\',summary.reindex(columns=failure_columns))\n        table=ils.copy();pend=summary.loc[summary.scope.eq(\'PENDIKOM_TRAIN\')].set_index(\'k\')\n        for col in [\'n_resolved_base_ls\',\'unresolved_cluster_dimensions\',\'unresolved_learner_rate\']:\n            table[col]=table.k.map(pend[col]) if col in pend else np.nan\n        for metric in s.METRIC_NAMES:\n            table[metric+\'_mean_ci95\']=table.apply(lambda r:f"{100*r[metric]:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]" if pd.notna(r[metric]) else \'NA (fit failed)\',axis=1)\n        s.save_csv(out/\'TABEL/01_sensitivitas_k_ILS.csv\',table[[\'k\',\'status\']+[m+\'_mean_ci95\' for m in s.METRIC_NAMES]+[\'coverage\',\'complete_profile_coverage\',\'n_resolved_base_ls\',\'unresolved_cluster_dimensions\']])\n        s.save_csv(out/\'TABEL/02_sensitivitas_k_stabilitas.csv\',stability)\n        s.save_csv(out/\'TABEL/03_diversitas_full_dan_train.csv\',summary.loc[summary.scope.isin([\'LMS_FULL\',\'PENDIKOM_TRAIN\'])])\n        figsig=s.digest({\'code\':code,\'ILS\':s.sha((out/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\'stability\':s.sha((out/\'LMS_STABILITAS/02_ringkasan_per_k.csv\').read_bytes())})\n        if not s.cache_read(files,control/\'gambar.json\',figsig,cfg[\'force\']):s.cache_write(files,control/\'gambar.json\',figsig,figures(s,out,ils,stability,summary),{})\n        s.unchanged(files,allwatched,checks);s.save_json(out/\'PROTOKOL/03_manifest_masukan.json\',allwatched)\n        failures=int((~summary.fit_valid.astype(bool)).sum())\n        status={\'status\':\'selesai\' if failures==0 else \'selesai_dengan_kegagalan_algoritmik\',\'completed_partitions\':completed,\n            \'fit_counts\':fit_counts,\'profile_counts\':map_counts,\'stability_counts\':stab_counts,\'n_valid_partitions\':planned-failures,\n            \'n_failed_partitions\':failures,\'n_ILS_valid_k\':len([k for k in cfg[\'k_values\'] if predictions[k] is not None]),\n            \'full_protocol\':cfg[\'k_values\']==GRID and cfg[\'resample_runs\']==30,\'k16_reuse_eligible\':can_reuse,\n            \'evaluation_cached\':cached,\'n_bootstrap\':int(data[\'valid_weights\'].sum()),\'primary_k\':16,\'q\':16,\n            \'k_selection\':\'none\',\'output\':str(out),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        manifest={str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):s.sha(p.read_bytes()) for p in out.rglob(\'*\') if p.is_file() and control not in p.parents}\n        s.save_json(control/\'manifest_keluaran.json\',manifest);s.save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 29 selesai:\',fit_counts,\'| profile:\',map_counts,\'| failures:\',failures,flush=True)\n        print(\'Hasil:\',out)\n        return {\'ils\':ils,\'table\':table,\'stability\':stability,\'status\':status,\'output\':str(out)}\n    except BaseException as error:\n        s.save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'completed_partitions\':completed,\'fit_counts\':fit_counts,\n            \'error\':type(error).__name__+\': \'+str(error),\'resume\':\'jalankan notebook 29 yang sama; cache diperiksa berdasarkan isi per partisi/profile/evaluasi\'})\n        raise\n'

_k29 = types.ModuleType('_k29')
exec(compile(K29_SOURCE, '_k29', 'exec'), _k29.__dict__)


In [5]:
HASIL_29 = _k29.run_sensitivity(ROOT_29, KONFIGURASI_29, K29_SOURCE,
                               SEMANTIC_K_SOURCE, _s26, _semantic_k)
display(HASIL_29['table'])
display(HASIL_29['stability'])
print(HASIL_29['status'])
print('Folder hasil:', HASIL_29['output'])


Grid k: [8, 12, 16, 20, 24] | q=16 | refinement bound=4 | expected partitions: 160
k=16: gunakan artefak terverifikasi tahap 25/26; fitting baru maksimum 128
LMS_FULL | rep= None | n= 6658
LMS_RESAMPLING | rep= 0 | n= 5326
LMS_RESAMPLING | rep= 1 | n= 5326
LMS_RESAMPLING | rep= 2 | n= 5326
LMS_RESAMPLING | rep= 3 | n= 5326
LMS_RESAMPLING | rep= 4 | n= 5326
LMS_RESAMPLING | rep= 5 | n= 5326
LMS_RESAMPLING | rep= 6 | n= 5326
LMS_RESAMPLING | rep= 7 | n= 5326
LMS_RESAMPLING | rep= 8 | n= 5326
LMS_RESAMPLING | rep= 9 | n= 5326
LMS_RESAMPLING | rep= 10 | n= 5326
LMS_RESAMPLING | rep= 11 | n= 5326
LMS_RESAMPLING | rep= 12 | n= 5326
LMS_RESAMPLING | rep= 13 | n= 5326
LMS_RESAMPLING | rep= 14 | n= 5326
LMS_RESAMPLING | rep= 15 | n= 5326
LMS_RESAMPLING | rep= 16 | n= 5326
LMS_RESAMPLING | rep= 17 | n= 5326
LMS_RESAMPLING | rep= 18 | n= 5326
LMS_RESAMPLING | rep= 19 | n= 5326
LMS_RESAMPLING | rep= 20 | n= 5326
LMS_RESAMPLING | rep= 21 | n= 5326
LMS_RESAMPLING | rep= 22 | n= 5326
LMS_RESAMPLING |

,k,dimension,n_test,valid_partition,status,n_bootstrap,coverage,complete_profile_coverage,accuracy,accuracy_ci95_low,...,balanced_accuracy_ci95_high,macro_f1,macro_f1_ci95_low,macro_f1_ci95_high,n_resolved_base_ls,unresolved_cluster_dimensions,unresolved_learner_rate,accuracy_mean_ci95,balanced_accuracy_mean_ci95,macro_f1_mean_ci95
0,8,mean_four_dimensions,139,True,VALID,10000,1.000000,1.000000,0.877698,0.845324,...,0.840668,0.806751,0.746580,0.854227,6,0,0.000000,"87.77 [84.53, 90.83]","78.84 [73.21, 84.07]","80.68 [74.66, 85.42]"
1,12,mean_four_dimensions,139,True,VALID,10000,1.000000,1.000000,0.881295,0.852518,...,0.839238,0.807548,0.753275,0.853231,6,0,0.000000,"88.13 [85.25, 90.83]","79.41 [74.69, 83.92]","80.75 [75.33, 85.32]"
2,16,mean_four_dimensions,139,True,VALID,10000,1.000000,1.000000,0.897482,0.870504,...,0.857619,0.829776,0.776050,0.872808,8,0,0.000000,"89.75 [87.05, 92.27]","81.29 [76.52, 85.76]","82.98 [77.61, 87.28]"
3,20,mean_four_dimensions,139,True,VALID,10000,0.998201,0.992806,0.861511,0.830935,...,0.833790,0.792940,0.733291,0.841312,8,1,0.024845,"86.15 [83.09, 89.03]","78.49 [73.37, 83.38]","79.29 [73.33, 84.13]"
4,24,mean_four_dimensions,139,True,VALID,10000,1.000000,1.000000,0.857914,0.825540,...,0.818626,0.773180,0.717032,0.822256,8,1,0.006211,"85.79 [82.55, 88.85]","77.34 [72.74, 81.86]","77.32 [71.70, 82.23]"


,k,n_planned,n_valid,n_failed,mean_direction_agreement_mean,mean_direction_agreement_sd,mean_direction_agreement_min,mean_direction_agreement_max,base_ls_agreement_mean,base_ls_agreement_sd,...,sensing_intuitive_agreement_min,sensing_intuitive_agreement_max,visual_verbal_agreement_mean,visual_verbal_agreement_sd,visual_verbal_agreement_min,visual_verbal_agreement_max,sequential_global_agreement_mean,sequential_global_agreement_sd,sequential_global_agreement_min,sequential_global_agreement_max
0,8,30,30,0,0.790010,0.025260,0.737890,0.839701,0.471711,0.061139,...,0.699962,0.875892,0.935330,0.024653,0.882088,0.974089,0.710596,0.042714,0.632370,0.819564
1,12,30,30,0,0.791476,0.017395,0.759857,0.822756,0.448780,0.048746,...,0.713669,0.812430,0.933678,0.026824,0.869696,0.969208,0.726956,0.043330,0.614345,0.800976
2,16,30,30,0,0.779930,0.018507,0.749108,0.822944,0.424365,0.050559,...,0.687007,0.818626,0.916128,0.032236,0.866692,0.975779,0.742609,0.026926,0.689072,0.802854
3,20,30,30,0,0.776214,0.019770,0.713293,0.821583,0.404900,0.065422,...,0.708975,0.811303,0.890744,0.013043,0.853173,0.916448,0.723864,0.039781,0.551633,0.792527
4,24,30,30,0,0.773765,0.016828,0.738922,0.832614,0.392102,0.054621,...,0.707848,0.804919,0.894999,0.013202,0.867255,0.926211,0.727150,0.031722,0.647766,0.782013


{'status': 'selesai', 'completed_partitions': 160, 'fit_counts': {'REUSED_25_26': 32, 'FIT': 128, 'CACHE': 0}, 'profile_counts': {'REUSED_26': 32, 'MAP': 128, 'CACHE': 0}, 'stability_counts': {'REUSED_26': 30, 'CALCULATE': 120, 'CACHE': 0, 'UNAVAILABLE': 0}, 'n_valid_partitions': 160, 'n_failed_partitions': 0, 'n_ILS_valid_k': 5, 'full_protocol': True, 'k16_reuse_eligible': True, 'evaluation_cached': False, 'n_bootstrap': 10000, 'primary_k': 16, 'q': 16, 'k_selection': 'none', 'output': '/content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/29_Sensitivitas_Jumlah_Cluster_K', 'finished_utc': '2026-11-06T22:55:00.037166+00:00'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/29_Sensitivitas_Jumlah_Cluster_K


## Membaca keluaran

- `TABEL/01_sensitivitas_k_ILS.csv`: mean accuracy/BA/F1 dan CI, coverage,
  jumlah Base LS dan unresolved pada Pendikom.
- `TABEL/02_sensitivitas_k_stabilitas.csv`: mean, SD, rentang dan jumlah
  replikasi valid per k; `03_diversitas_full_dan_train.csv` untuk profil.
- `GAMBAR/`: kurva k–ILS (CI) dan k–stabilitas LMS (SD), PNG/PDF.
- `PARTISI/`, `PROFIL/`: mode, assignment, mapping dan bukti semantik per k.
  Untuk sumber k16 yang dipakai ulang, lokasi aslinya tercantum pada peta
  artefak; tidak dibuat salinan input terpisah.
- `LMS_STABILITAS/`: hasil per replikasi, ringkasan, dan selisih berpasangan.
- `ILS/`: prediksi, counts/confusion matrices, metrik, bank bootstrap, serta
  selisih berpasangan vs k16 dan interval keluarga empat untuk mean BA.
- `PROTOKOL/`: parameter, sumber, peta artefak, indeks SI, diversitas dan
  perbandingan lintas-k. `AUDIT/` mencatat validitas fitting.
- `KONTROL/`: cache isi, status dan manifest hasil, khusus tahap 29.

Seluruh nilai k dilaporkan sebagai analisis sensitivitas dengan acuan k=16.
